> `oeai_mod_arbor_gold.ipynb`
> [20251105.1]
> *Arbor Gold notebook*

In [0]:
%run ./oeai_mod_arbor_env_var

In [0]:
# Initialise Logging
oeai.notebook = SimpleNamespace(
    name = "oeai_mod_arbor_gold.ipynb",
    buildversion = "20251105.1",
    buildtimestamp = "2025-11-05T16:00:00Z",
)
oeai.log.init()
oeai.log.start_block("Notebook Init")

In [0]:
from pyspark.sql import DataFrame
from typing import Tuple
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import IntegerType, StringType, ArrayType, StructType, StructField, DateType
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, month, dayofmonth
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

In [0]:
# List of Delta table names to process
delta_tables = [
    "dim_Organisation", 
    "dim_Date",
    "dim_Student", 
    "dim_StudentExtended", 
    "fact_AttendanceSummary", 
    "fact_AttendanceSession", 
    "fact_Exclusion",
    "dim_Group",
    "dim_GroupMembership",
    "fact_Behaviour",
    "fact_Achievement",
    "dim_ExclusionReason",
    "dim_SENDNeed",
    "fact_Attainment",
    "dim_Staff",
    "fact_StaffAbsence",
    "fact_AttendanceLesson",
    "fact_CourseEnrolment",
    "dim_UserDefinedFields",
    "fact_Detention",
    "fact_PointAward",
    "fact_InternalExclusion",
    "fact_ExamResult",
]

# Functions
These functions are all used in the processing block

In [0]:
oeai.log.start_block("Functions")

In [0]:
from pyspark.sql import DataFrame, functions as F, types as T

def ensure_academic_year_column(
    df: DataFrame,
    date_column: str,
    school_id_column: str | None = None,   # optional
    input_format: str = "yyyy-MM-dd",      # if string, parse with this format
    start_month: int = 8,                  # August
    dim_years_subpath: str = "dim_AcademicYear"  # joined to silver_path
) -> DataFrame:
    """
    Ensures `date_column` is cast to DateType and adds 'Academic_Year' column like '2024/2025'.

    Priority logic:
      1) If `school_id_column` is provided and present in df, try to look up the
         academic year from the dimension table (loaded internally from
         `silver_path + dim_years_subpath`).
      2) If no match (or any issue occurs), fall back to month-based academic-year logic.
    """
    if date_column not in df.columns:
        raise ValueError(f"Column '{date_column}' not found in DataFrame.")

    # Always cast column to proper date
    df = df.withColumn(date_column, F.to_date(F.col(date_column), input_format))

    # Prepare fallback (old) logic
    start_year = F.when(
        F.month(F.col(date_column)) >= start_month,
        F.year(F.col(date_column))
    ).otherwise(F.year(F.col(date_column)) - F.lit(1))
    end_year = start_year + F.lit(1)
    fallback_ay = F.concat(start_year.cast(T.StringType()), F.lit("/"), end_year.cast(T.StringType()))

    # If no school_id info, just return fallback immediately
    if not school_id_column or school_id_column not in df.columns:
        # print("[ensure_academic_year_column] 📝 Using FALLBACK logic (no school_id provided).")
        oeai.log.info("Using FALLBACK logic (no school_id provided).", function="ensure_academic_year_column")
        return df.withColumn("Academic_Year", fallback_ay)

    try:
        # Load and deduplicate dim table
        dim_years_path = silver_path + dim_years_subpath
        dim = (
            spark.read.format("delta").load(dim_years_path)
                .select(
                    F.col("school_id").alias("_dim_school_id"),
                    F.col("CALENDAR_YEAR_START_DATE").alias("_dim_start"),
                    F.col("CALENDAR_YEAR_END_DATE").alias("_dim_end"),
                    F.col("Academic_Year").alias("_dim_academic_year")
                )
                .dropDuplicates(["_dim_school_id", "_dim_academic_year"])
        )

        cond = (
            (F.col(school_id_column) == F.col("_dim_school_id")) &
            (F.col(date_column) >= F.col("_dim_start")) &
            (F.col(date_column) <= F.col("_dim_end"))
        )

        joined = df.join(F.broadcast(dim), on=cond, how="left")

        result = (
            joined.withColumn(
                "Academic_Year",
                F.coalesce(F.col("_dim_academic_year"), fallback_ay)
            )
            .drop("_dim_school_id", "_dim_start", "_dim_end", "_dim_academic_year")
        )

        # print("[ensure_academic_year_column] ✅ SUCCESS: Using DIMENSION logic (Academic_Year pulled from dim table).")
        oeai.log.info("SUCCESS: Using DIMENSION logic (Academic_Year pulled from dim table).", function="ensure_academic_year_column")
        return result

    except Exception as e:
        # print(f"[ensure_academic_year_column] 📝 NOTE: Falling back to month-based logic. Details: {e}")
        oeai.log.exception("Falling back to month-based logic", function="ensure_academic_year_column")
        return df.withColumn("Academic_Year", fallback_ay)


In [0]:
# Function to generate student_ext_academic_year

from pyspark.sql import DataFrame

def add_student_ext_academic_year_key(df: DataFrame) -> DataFrame:
    try:
        # Load dim_StudentExtendedAcademicYear with only the necessary columns
        dim_student_academic_year_group = spark.read.parquet(f"{gold_path}/dim_StudentExtendedAcademicYear").select(
            "studentkey", "Academic_Year", "studentextendedacademicyearkey"
        )
        # Check if the required columns exist in the input DataFrame
        if "studentkey" not in df.columns or "Academic_Year" not in df.columns:
            raise ValueError("The DataFrame must contain both 'studentkey' and 'Academic_Year' columns for the join.")

        # Join the fact table with dim_StudentExtendedAcademicYear on studentkey and Academic_Year
        df_with_key = df.join(
            dim_student_academic_year_group,
            on=["studentkey", "Academic_Year"],
            how="left"
        )

        return df_with_key

    except ValueError as ve:
        # print(f"ValueError: {ve}")
        oeai.log.exception("ValueError", function="add_student_ext_academic_year_key")
        raise  # Re-raise the exception after logging

    except Exception as e:
        # print(f"An error occurred while adding student academic year group key: {e}")
        oeai.log.exception("An error occurred while adding student academic year group key", function="add_student_ext_academic_year_key")
        raise  # Re-raise the exception after logging

In [0]:
def add_student_academic_year_group_key(df: DataFrame) -> DataFrame:
    # Load dim_StudentAcademicYearGroup with only the necessary columns
    dim_student_academic_year_group = spark.read.parquet(f"{gold_path}/dim_StudentAcademicYearGroup").select(
        "studentkey", "Academic_Year", "studentacademicyeargroupkey"
    )

    # Join the fact table with dim_StudentAcademicYearGroup on studentkey and Academic_Year
    df_with_key = df.join(
        dim_student_academic_year_group,
        on=["studentkey", "Academic_Year"],
        how="left"
    )

    return df_with_key

In [0]:
# Function to map age to year group
def get_year_group_from_age(age):
    year_group_mapping = {
        0: -4, 1: -3, 2: -2, 3: -1, 4: 0,
        5: 1, 6: 2, 7: 3, 8: 4, 9: 5,
        10: 6, 11: 7, 12: 8, 13: 9, 14: 10,
        15: 11, 16: 12, 17: 13, 18: 14
    }
    return year_group_mapping.get(age, None)

In [0]:
# def generate_academic_years_and_groups

from datetime import datetime, date

def generate_academic_years_and_groups(Current_YG, current_academic_year, leaving_date, years_back=4):
    """
    Returns a list of tuples: (academic_year_str, 'Year N', mapped_label)

    Rules:
    - If leaving_date is None: treat as still enrolled; Current_YG is for current_academic_year.
    - If they left before the current academic year started: Current_YG is the YG in their *last enrolled*
      academic year; do not decrement relative to the *current* year—decrement relative to that leaving year.
    """
    if Current_YG is None or not current_academic_year:

        return []

    # Parse the current academic year's start (YYYY/ZZZZ -> YYYY)
    start_year = int(str(current_academic_year).split('/')[0])
    current_year_start = date(start_year, 8, 1)

    # Normalize leaving_date
    if isinstance(leaving_date, datetime):
        ld = leaving_date.date()
    else:
        ld = leaving_date  # may be a date or None

    # Used only to decide which academic years to include in the output
    ld_effective = ld or date.today()

    # Determine which academic year the given Current_YG actually refers to (the "base" year)
    # - If the pupil left before the current academic year started, freeze at their leaving academic year.
    # - Otherwise, they're still enrolled for the current year; base is the current academic year.
    if ld is not None and ld < current_year_start:
        leaving_start_year = ld.year if ld.month >= 8 else ld.year - 1
        base_start_year = leaving_start_year
    else:
        base_start_year = start_year

    # Year Group mapping
    year_group_mapping = {
        -4: "E1", -3: "E2", -2: "N1", -1: "N2",
         0: "R", 1: "1", 2: "2", 3: "3", 4: "4", 5: "5", 6: "6",
         7: "7", 8: "8", 9: "9", 10: "10", 11: "11",
        12: "12", 13: "13", 14: "14"
    }

    prior_years = []
    for i in range(years_back):
        ay_start_year = start_year - i
        ay_label = f"{ay_start_year}/{ay_start_year + 1}"
        ay_start_date = date(ay_start_year, 8, 1)

        # Only include academic years that began on/before the leaving date (or "today" if still enrolled)
        if ld_effective >= ay_start_date:
            # Decrement relative to the *base* year (current if still enrolled, leaving year if left earlier)
            years_between = base_start_year - ay_start_year  # 0 for base year, 1 for year before, etc.
            adjusted_year_group = Current_YG - years_between

            std_year_group_str = year_group_mapping.get(adjusted_year_group, "Unknown")
            prior_years.append((
                ay_label,
                f"Year {adjusted_year_group}",
                std_year_group_str
            ))

    return prior_years


In [0]:
# UDF to return std_year_group or fall back to a calculation based on dob
def calculate_age_or_year_group(dob, academic_year, std_Year_Group):
    if std_Year_Group is not None:
        # Use the value of Year_Group
        return std_Year_Group
    elif dob is not None:
        # Calculate age based on Date_Of_Birth
        start_year = int(academic_year.split('/')[0])
        aug_31 = datetime(start_year, 8, 31)
        age = aug_31.year - dob.year - ((aug_31.month, aug_31.day) < (dob.month, dob.day))
        # subtract 4 to adjust for starting year group
        return (age - 4)
    else:
        # Return None if both Date_Of_Birth and Year_Group are missing
        return None


In [0]:
# def get_reference_path
def get_reference_path(gold_path: str, file_name: str) -> str:
    try:
        # If oeai.path.reference exists and is valid, use it
        ref_base = oeai.path.reference
        if not ref_base:  # catch empty string or None
            raise AttributeError
    except (AttributeError, NameError):
        # Derive from gold_path by replacing the last folder with "reference"
        parent = os.path.dirname(gold_path.rstrip("/"))
        ref_base = os.path.join(parent, "reference")

    return os.path.join(ref_base, file_name)

In [0]:
# Declare UDFs
# UDF for age
calculate_age_udf = F.udf(calculate_age_or_year_group, IntegerType())

# UDF for year group mapping
year_group_udf = F.udf(get_year_group_from_age, IntegerType())

# UDF for generating prior academic years and Year Groups
generate_prior_years_udf = F.udf(generate_academic_years_and_groups, ArrayType(StructType([
    StructField("AcademicYear", StringType(), True),
    StructField("AdjustedYearGroup", StringType(), True),
    StructField("std_year_group", StringType(), True)
])))

In [0]:
oeai.log.end_block(index=0, include_target=True)
oeai.log.checkpoint("Init")

# Processing calculated Gold tables

In [0]:
oeai.log.start_block("Processing")

##### Create dim_StudentAcademicYearGroup.  Hosts each academic year entry for the student along with their year group.  This is done before the main processing as the StudentAcademicYearGroupkey will be added to the fact tables

In [0]:
oeai.log.start_block("Create dim_StudentAcademicYearGroup")

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

# Load dim_Student from the silver path in Delta format
dim_student = spark.read.format("delta").load(f"{silver_path}/dim_Student")

# Load dim_StudentExtended from the silver path in Delta format
dim_studentex = spark.read.format("delta").load(f"{silver_path}/dim_StudentExtended")

# Load dim_NCYearGroup from reference folder
reference_path = get_reference_path(gold_path, "dim_NCYearGroup.csv")
dim_NCYearGroup = spark.read.csv(reference_path, header=True, inferSchema=True)

# Join dim_student with dim_studentex on studentkey
dim_student = dim_student.join(
    dim_studentex.select("studentkey", "Admission_Date", "Leaving_Date", "Year_Group"),
    on="studentkey",
    how="left"
)

# Join dim_student with dim_NCYearGroup on Year_Group_Description
dim_student = dim_student.join(
    dim_NCYearGroup,
    dim_student["Year_Group"] == dim_NCYearGroup["Year_Group_Description"],
    "left"
)

# Select necessary columns
dim_student_selected = dim_student.select(
    F.col("studentkey"),
    F.col("school_id"),
    F.to_date(F.col("Date_Of_Birth"), "yyyy-MM-dd").alias("Date_Of_Birth"),
    F.to_date(F.col("Admission_Date"), "yyyy-MM-dd").cast(DateType()).alias("Admission_Date"),
    F.to_date(F.col("Leaving_Date"), "yyyy-MM-dd").cast(DateType()).alias("Leaving_Date"),
    F.col("std_Year_Group").alias("std_Year_Group")
)

# === NEW: Build SAYG rows from actual enrolment + YEAR membership at January census ===

from pyspark.sql import functions as F, Window
from pyspark.sql.types import DateType

# 1) Load sources we need for SEA-like logic
saye_df = spark.read.format("delta").load(f"{silver_path}/dim_StudentAcademicYearEnrolment") \
    .select("studentkey", "Academic_Year").dropDuplicates()


# We'll need school_id for later joins; we already have it in dim_student_selected
students_base = dim_student_selected.select("studentkey","school_id").dropDuplicates()

# 2) Compute January census date (3rd Thursday) per AY
#    Reuse your third_thursday() approach as a UDF so it matches SEA behaviour
def third_thursday(year: int):
    jan1 = datetime(year, 1, 1)
    days_to_thu = (3 - jan1.weekday()) % 7
    return (jan1 + timedelta(days=days_to_thu + 14)).date()

def safe_third_thursday(academic_year):
    # Accept None or malformed AY; return None in that case
    if academic_year is None:
        return None
    s = str(academic_year).strip()
    # Match 'YYYY/YY' or 'YYYY/YYYY' with optional spaces
    m = re.match(r'^(\d{4})\s*/\s*(\d{2,4})$', s)
    if not m:
        return None
    start = int(m.group(1))
    end = int(m.group(2))
    # Expand two-digit end years using the century of the start year (e.g., 2024/25 -> 2025)
    if end < 100:
        end = (start // 100) * 100 + end
    return third_thursday(end)

safe_third_thursday_udf = F.udf(safe_third_thursday, DateType())

census_df = saye_df.select("studentkey","Academic_Year") \
    .withColumn("Census_Date", safe_third_thursday_udf(F.col("Academic_Year")))


# 3) YEAR memberships; parse dates and attach Year_Group text

from pyspark.sql import functions as F

dim_group_year = (
    spark.read.format("delta").load(f"{silver_path}/dim_Group")
    .select("groupkey", "Group_Type", "Group_Description", "Group_Name")
    .withColumn("Group_Type_norm", F.upper(F.trim(F.col("Group_Type"))))
    .filter(F.col("Group_Type_norm") == F.lit("YEAR"))
)

gm_year = (
    spark.read.format("delta").load(f"{silver_path}/dim_GroupMembership")
    .select("studentkey", "groupkey",
            F.to_date(F.col("Start_Date")).alias("Start_Date"),
            F.to_date(F.col("End_Date")).alias("Membership_End_Date"))
    .join(dim_group_year, on="groupkey", how="inner")
    # Prefer Group_Description; fall back to Group_Name
    .withColumn("Year_Group", F.coalesce(F.col("Group_Description"), F.col("Group_Name")))
    .select("studentkey", "Year_Group", "Start_Date", "Membership_End_Date")
)



# 4) For each (student, AY), pick YEAR membership active on census,
#    else latest membership that started before census
#    else latest memberbership within that academic year
from pyspark.sql import functions as F, Window

# Base + AY end date (31 July of the 2nd year), no Python UDFs needed
base = (
    census_df.join(students_base, "studentkey", "left")
    .withColumn("end_year", F.split(F.col("Academic_Year"), "/").getItem(1).cast("int"))
    .withColumn("AY_End_Date", F.make_date(F.col("end_year"), F.lit(7), F.lit(31)))
    .drop("end_year")
)

# 1) Best: membership active on census
active_on_census = (
    base.join(gm_year, "studentkey", "left")
        .filter(
            (F.col("Start_Date") <= F.col("Census_Date")) &
            (F.col("Membership_End_Date").isNull() | (F.col("Membership_End_Date") >= F.col("Census_Date")))
        )
        .select("studentkey","Academic_Year","Year_Group","school_id")
        .withColumn("pref", F.lit(1))
)

# 2) Fallback: latest that started before/on census
last_before_census = (
    base.join(gm_year, "studentkey", "left")
        .filter(F.col("Start_Date") <= F.col("Census_Date"))
        .withColumn(
            "rn",
            F.row_number().over(
                Window.partitionBy("studentkey","Academic_Year")
                      .orderBy(F.col("Start_Date").desc())
            )
        )
        .filter(F.col("rn")==1)
        .select("studentkey","Academic_Year","Year_Group","school_id")
        .withColumn("pref", F.lit(2))
)

# 3) NEW fallback: first membership that starts AFTER census but BEFORE/ON 31 July of that AY
first_after_census = (
    base.join(gm_year, "studentkey", "left")
        .filter(
            (F.col("Start_Date") > F.col("Census_Date")) &
            (F.col("Start_Date") <= F.col("AY_End_Date"))
        )
        .withColumn(
            "rn",
            F.row_number().over(
                Window.partitionBy("studentkey","Academic_Year")
                      .orderBy(F.col("Start_Date").asc())
            )
        )
        .filter(F.col("rn")==1)
        .select("studentkey","Academic_Year","Year_Group","school_id")
        .withColumn("pref", F.lit(3))
)

# Prefer 1, else 2, else 3 (deterministic tie-break on Year_Group)
yg_on_census = (
    active_on_census.unionByName(last_before_census).unionByName(first_after_census)
    .withColumn(
        "rowpick",
        F.row_number().over(
            Window.partitionBy("studentkey","Academic_Year")
                  .orderBy(F.col("pref").asc(), F.col("Year_Group").asc())
        )
    )
    .filter(F.col("rowpick")==1)
    .drop("pref","rowpick")
)


# 5) Keep only AYs the pupil was actually enrolled in (SAYE), same as SEA
#    + robust mapping to std_Year_Group via a normalised join
from pyspark.sql import functions as F

# Aliases (helps readability; not strictly required)
saye_df = saye_df.alias("saye")
yg_on_census = yg_on_census.alias("yg")

# Join SAYE with the YEAR group-at-census result (yg already includes school_id)
sayg_enrolled = (
    saye_df
    .join(yg_on_census, ["studentkey", "Academic_Year"], "left")
)

# Normalise Year_Group labels and join to dim_NCYearGroup safely (trim/case-insensitive)
dim_NCYearGroup_norm = dim_NCYearGroup.select(
    F.upper(F.trim(F.col("Year_Group_Description"))).alias("YG_DESC_NORM"),
    F.col("std_Year_Group").cast("int").alias("std_Year_Group")
)

sayg_enrolled = sayg_enrolled.withColumn(
    "YG_NORM", F.upper(F.trim(F.col("Year_Group")))
)

sayg_enrolled = sayg_enrolled.join(
    dim_NCYearGroup_norm,
    F.col("YG_NORM") == F.col("YG_DESC_NORM"),
    "left"
).drop("YG_NORM", "YG_DESC_NORM")

# Final shape (no ambiguous school_id now—only from yg_on_census)
sayg_enrolled = (
    sayg_enrolled
    .select(
        "studentkey",
        "school_id",
        "Academic_Year",
        "Year_Group",
        F.col("std_Year_Group").alias("std_year_group")
    )
    .dropDuplicates(["studentkey", "Academic_Year"])
)



# 6) Compulsory school age flag (retain your existing logic, keyed off 31-Aug of AY)
sayg_enriched = sayg_enrolled.withColumn(
    "Ref_Date",
    F.to_date(F.concat(F.substring(F.col("Academic_Year"), 1, 4), F.lit("-08-31")))
).join(
    dim_student_selected.select("studentkey","Date_Of_Birth"), "studentkey", "left"
)

sayg_enriched = sayg_enriched.withColumn(
    "Age",
    F.year(F.col("Ref_Date")) - F.year(F.col("Date_Of_Birth")) -
    F.when(F.date_format(F.col("Ref_Date"), "MM-dd") < F.date_format(F.col("Date_Of_Birth"), "MM-dd"), 1).otherwise(0)
).withColumn(
    "CompulsorySchoolAge",
    F.when((F.col("Age") >= 5) & (F.col("Age") <= 15), 1).otherwise(0)
).drop("Ref_Date","Age")

# 7) Provide 'final_df' in the same shape your downstream expects
final_df = sayg_enriched


# Add a monotonically increasing ID and adjust it to start from 1
final_df_with_key = final_df.withColumn(
    "studentacademicyeargroupkey",
    F.row_number().over(Window.orderBy(F.monotonically_increasing_id()))
)

### Filter dim_StudentAcademicYearGroup to exclude students without an attendance record in that academic year

# Load AttendanceSession from the silver path in Delta format
df_as = spark.read.format("delta").load(f"{silver_path}/fact_AttendanceSession")
from pyspark.sql.functions import col, to_date, lit, concat, substring, date_add, sum as F_sum, when, trim, lower

# Step 1: Get academic year start or fall back (1st Aug) and end (31st Jul next year) dates in final_df_with_key
from pyspark.sql import functions as F

# --- fallback pieces (same as your existing logic) ---
fallback_start = F.to_date(F.concat(F.substring(F.col("Academic_Year"), 1, 4), F.lit("-08-01")), "yyyy-MM-dd")
fallback_end   = F.date_add(fallback_start, 364)  # same as your current code

try:
    # Load and deduplicate the dimension by (school_id, Academic_Year)
    dim_years_path = silver_path + "dim_AcademicYear"
    df_dim_acyear = (
        spark.read.format("delta").load(dim_years_path)
        .select(
            "school_id",
            "Academic_Year",
            F.col("CALENDAR_YEAR_START_DATE").alias("_dim_start"),
            F.col("CALENDAR_YEAR_END_DATE").alias("_dim_end")
        )
        .dropDuplicates(["school_id", "Academic_Year"])
    )

    # Left join on school_id + Academic_Year (broadcast is typically fine for small dims)
    joined = final_df_with_key.join(F.broadcast(df_dim_acyear), on=["school_id", "Academic_Year"], how="left")

    # Use dim dates when present, otherwise fallback (coalesce)
    final_df_with_key = (
        joined
        .withColumn("academic_year_start", F.coalesce(F.col("_dim_start"), fallback_start))
        .withColumn("academic_year_end",   F.coalesce(F.col("_dim_end"),   fallback_end))
        .drop("_dim_start", "_dim_end")
    )

    # print("[academic-year dates] 🎉 SUCCESS: Using DIMENSION dates when available; falling back per-row when missing.")
    oeai.log.info("[academic-year dates] SUCCESS: Using DIMENSION dates when available; falling back per-row when missing.")

except Exception as e:
    # If anything fails above, revert to pure fallback for all rows
    final_df_with_key = (
        final_df_with_key
        .withColumn("academic_year_start", fallback_start)
        .withColumn("academic_year_end",   fallback_end)
    )
    # print(f"[academic-year dates] 📝 NOTE: Could not use dimension dates; using FALLBACK for all rows. Details: {e}")
    oeai.log.exception(f"[academic-year dates] 📝 NOTE: Could not use dimension dates; using FALLBACK for all rows.")

# Step 2: Filter AttendanceSession by academic year range
df_as_filtered_by_year = df_as.join(
    final_df_with_key.select("studentkey", "Academic_Year", "academic_year_start", "academic_year_end"),
    on="studentkey",
    how="inner"
).filter(
    (col("Date") >= col("academic_year_start")) &  # Date falls in academic year
    (col("Date") <= col("academic_year_end"))  # Date falls in academic year
)

# Step 3: Identify students who only have irrelevant marks (#, Z) or missing marks (-, #, Z)
students_to_exclude = df_as_filtered_by_year.groupBy("studentkey", "Academic_Year").agg(
    F_sum(when(col("Mark").isin("#", "Z", "-"), 1).otherwise(0)).alias("sum_irrelevant_marks"),
    F_sum(when(~col("Mark").isin("#", "Z", "-"), 1).otherwise(0)).alias("sum_valid_marks")
).filter(
    (col("sum_irrelevant_marks") > 0) &  # At least one irrelevant or missing mark
    (col("sum_valid_marks") == 0)  # No valid marks at all
).select("studentkey", "Academic_Year")

# Step 4: Exclude students identified in Step 3
df_as_valid = df_as_filtered_by_year.join(
    students_to_exclude, on=["studentkey", "Academic_Year"], how="leftanti"
)

# Step 5: Drop duplicates based on studentkey and academic year to ensure unique rows
filtered_df = df_as_valid.dropDuplicates(["studentkey", "Academic_Year"])

# Step 6: Join back to final_df_with_key to retain valid records only
result_df = final_df_with_key.join(
    filtered_df.select("studentkey", "Academic_Year").distinct(),
    on=["studentkey", "Academic_Year"],
    how="inner"
)

# Step 7: Drop the academic year start and end columns
result_df_cleaned = result_df.drop("academic_year_start", "academic_year_end")

# OnRollOnCensus
# Assuming final_path is already defined
path = f"{silver_path}/dim_StudentExtended"

# Read the Delta table
df_se = spark.read.format("delta").load(path).select(
    "studentkey",
    F.col("Admission_Date").cast(DateType()).alias("Admission_Date"),
    F.col("Leaving_Date").cast(DateType()).alias("Leaving_Date")
)
# Assuming dim_student and result_df_cleaned are PySpark DataFrames

# Perform the join
result_df_cleaned = result_df_cleaned.join(
    df_se.select("studentkey", "Admission_Date", "Leaving_Date"), 
    on="studentkey", 
    how="left"
)

from pyspark.sql import functions as F
from pyspark.sql.types import DateType
import calendar
from datetime import datetime, timedelta

# Function to calculate the 3rd Thursday in January for a given year
def third_thursday(year):
    # January 1st of the year
    january_first = datetime(year, 1, 1)
    # Get the weekday of January 1st (0 = Monday, 1 = Tuesday, ..., 6 = Sunday)
    weekday_of_jan_first = january_first.weekday()
    # Calculate the date of the 3rd Thursday
    days_to_thursday = (3 - weekday_of_jan_first) % 7  # Find the first Thursday
    third_thursday_date = january_first + timedelta(days=days_to_thursday + 14)  # Add 14 more days to get the 3rd Thursday
    return third_thursday_date.date()  # Return the date in YYYY-MM-DD format

# Register the UDF
def third_thursday_udf(academic_year):
    year = int(academic_year.split('/')[1])  # Extract year after "/"
    return third_thursday(year)

# Register the function as a UDF
third_thursday_udf_spark = F.udf(third_thursday_udf, DateType())

# Add census date
result_df_cleaned = result_df_cleaned.withColumn(
    "Census_Date", safe_third_thursday_udf(F.col("Academic_Year"))
)


# Show the updated DataFrame
#result_df_cleaned.show(truncate=False)

from pyspark.sql import functions as F

# Add OnRollOnCensus

result_df_cleaned = result_df_cleaned.withColumn(
    "OnRollOnCensus",
    F.when(
        (F.col("Admission_Date").isNull()), 
        "Unknown"
    ).when(
        (F.col("Admission_Date") <= F.col("Census_Date")) & 
        ((F.col("Leaving_Date") >= F.col("Census_Date")) | F.col("Leaving_Date").isNull()),
        1
    ).otherwise(0)
)
# Remove unnecessary columns
result_df_cleaned_dropped = result_df_cleaned.drop("Admission_Date", "Leaving_Date", "Census_Date")

# Write the result to a new parquet file
final_path = f"{gold_path}/dim_StudentAcademicYearGroup"
result_df_cleaned_dropped.write.mode("overwrite").parquet(final_path)

oeai.log.end_block()
oeai.log.checkpoint()

# Calculated Gold tables

##### Create dim_StudentCalculated. This section is to create a dim_StudentCalculated to host a number of calculated measures relevant to analytics, including:
       1. is_sen and Is_SEN_String (SEN or Not SEN)
       2. is_summerborn and Is_Summer_Born_String (Summer Born or Not Summer Born)
       3. is_current and Is_Current_String (Current or Not Current)
       4. Full name field (concat sur, for)
       5. PP eligable string
       6. EAL string

In [0]:
oeai.log.start_block("Create dim_StudentCalculated")

# Define the schema for the dim_StudentCalculated table
dim_student_calculated_schema = StructType([
    StructField("studentkey", StringType(), True),
    StructField("is_sen", IntegerType(), True),
    StructField("Is_SEN_String", StringType(), True),
    StructField("Is_PP_String", StringType(), True),
    StructField("Is_EAL_String", StringType(), True),
    StructField("is_summer_born", IntegerType(), True),
    StructField("Is_Summer_Born_String", StringType(), True),
    #StructField("is_current", IntegerType(), True),
    StructField("Is_Current_String", StringType(), True),
    StructField("Full_Name", StringType(), True),
    #StructField("is_in_year_joiner", IntegerType(), True),
    #StructField("Is_In_Year_Joiner_String", StringType(), True),
])

# Load the student and studentextended tables as Delta from silver_path
silver_student_dir = '/dim_Student'
silver_student_path = f"{silver_path}{silver_student_dir}/"
dim_student_df = (
    spark
      .read
      .format("delta")
      .load(silver_student_path)
)

silver_studentextended_dir = '/dim_StudentExtended'
silver_studentextended_path = f"{silver_path}{silver_studentextended_dir}/"
dim_studentextended_df = (
    spark
      .read
      .format("delta")
      .load(silver_studentextended_path)
)

# Calculate the is_summer_born and Is_Summer_Born_String columns
dim_student_df = dim_student_df.withColumn(
    "is_summer_born",
    when(
        (month(col("Date_Of_Birth")).between(4, 8)) &
        ((month(col("Date_Of_Birth")) > 4) | (month(col("Date_Of_Birth")) == 4) & (dayofmonth(col("Date_Of_Birth")) >= 1)) &
        ((month(col("Date_Of_Birth")) < 8) | (month(col("Date_Of_Birth")) == 8) & (dayofmonth(col("Date_Of_Birth")) <= 31)),
        1
    ).otherwise(0)
).withColumn(
    "Is_Summer_Born_String",
    when(col("is_summer_born") == 1, "Summer Born").otherwise("Not Summer Born")
)

# Add is_sen, Is_SEN_String, is_current, and Is_Current_String based on dim_studentextended_df
dim_studentextended_df = dim_studentextended_df.withColumn(
    "is_sen",
    when(col("SEN_Status").isin("K", "E"), 1).otherwise(0)
).withColumn(
    "Is_SEN_String",
    when(col("SEN_Status").isin("K", "E"), "SEN").otherwise("Not SEN")
).withColumn(
    "Is_PP_String",
    when(col("Pupil_Premium_Indicator") == True, "PP").otherwise("Not PP")
).withColumn(
    "Is_EAL_String",
    when(col("English_As_Additional_Language") == True, "EAL").otherwise("Not EAL")
).withColumn(
    "Is_Current_String",
    when(col("Is_Current") == True, "On Roll").otherwise("Leaver")
)

# Add Full_Name
dim_student_df = dim_student_df.withColumn(
    "Full_Name",
    F.concat(
        F.col("Legal_Surname"),
        F.lit(", "),  # Adds a comma and a space between the surname and forename
        F.col("Legal_Forename")
    )
)

# Join dim_student_df with dim_studentextended_df
dim_student_calculated_df = dim_student_df.join(dim_studentextended_df, "studentkey", "left").select(
    col("studentkey"),
    col("is_sen"),
    col("Is_SEN_String"),
    col("Is_PP_String"),
    col("Is_EAL_String"),
    col("is_summer_born"),
    col("Is_Summer_Born_String"),
    #col("is_current"),
    col("Is_Current_String"),
    col("Full_Name"),
    #col("is_in_year_joiner"),
    #col("Is_In_Year_Joiner_String")
)

# Ensure that the DataFrame conforms to the schema
dim_student_calculated_df = spark.createDataFrame(dim_student_calculated_df.rdd, dim_student_calculated_schema)

# Write the result to a new parquet file
dim_student_calculated_df.write.mode("overwrite").format("parquet").save(gold_path + "/dim_StudentCalculated")

oeai.log.end_block()
oeai.log.checkpoint()

##### Create dim_StudentExtendedAcademicYear

In [0]:
oeai.log.start_block("Create dim_StudentExtendedAcademicYear")

try:
    from pyspark.sql import functions as F
    from pyspark.sql.window import Window
    import datetime

    # -------------------------------------------------------------------------
    # 1. Load tables (now also includes dim_StudentCalculated & dim_StudentExtended)
    # -------------------------------------------------------------------------

    table_dirs = {
        'student': '/dim_Student',
        'group': '/dim_Group',
        'groupmembership': '/dim_GroupMembership',
        'student_extended_history': '/dim_StudentExtendedHistory',
        'date': '/dim_Date',
        'saye': '/dim_StudentAcademicYearEnrolment',
        # NEW / ADAPT: 'studcalc' => dim_StudentCalculated
        'studcalc': '/dim_StudentCalculated',
        # NEW / ADAPT: 'studext' => dim_StudentExtended
        'studext': '/dim_StudentExtended',
    }

    tables = {}

    for key, directory in table_dirs.items():
        path = (
            f"{gold_path if key in ['studcalc'] else silver_path}{directory}/"
        )
        tables[key] = spark.read.format(
            "delta" if key in ['student_extended_history', 'date', 'saye', 'student', 'group', 'groupmembership', 'studext'] else "parquet"
        ).load(path)

    student_df = tables['student']
    group_df = tables['group']
    groupmembership_df = tables['groupmembership']
    student_extended_history_df = tables['student_extended_history']
    date_df = tables['date']
    saye_df = tables['saye']
    # The newly mentioned tables:
    student_calculated_df = tables['studcalc']   # Contains Is_EAL_String, Is_PP_String, Is_SEN_String
    student_extended_2_df = tables['studext']    # Contains Ethnicity, SEN_Status (naming example)

    # -------------------------------------------------------------------------
    # 2. Format FullDate in dim_Date
    # -------------------------------------------------------------------------

    date_df = date_df.withColumn("FullDate", F.to_date(F.col("FullDate"), "dd/MM/yyyy"))

    # Get current and prior three academic years
    today = datetime.date.today()
    current_year = today.year
    current_month = today.month

    if current_month >= 8:
        current_academic_year = f"{current_year}/{current_year + 1}"

    else:
        current_academic_year = f"{current_year - 1}/{current_year}"

    academic_years = [
        current_academic_year,
        f"{int(current_academic_year.split('/')[0]) - 1}/{int(current_academic_year.split('/')[1]) - 1}",
        f"{int(current_academic_year.split('/')[0]) - 2}/{int(current_academic_year.split('/')[1]) - 2}"
        ]

    thursdays_in_january_df = (
        date_df
        .filter(
            (F.col("MonthNumberOfYear") == 1) &
            (F.col("DayName") == "Thursday") &
            (F.col("AcademicYear").isin(academic_years))
        )
        .select(F.col("AcademicYear").alias("Academic_Year"), "FullDate")
    )

    window_spec = Window.partitionBy("Academic_Year").orderBy("FullDate")

    third_thursday_df = (
        thursdays_in_january_df
        .withColumn("WeekRank", F.row_number().over(window_spec))
        .filter(F.col("WeekRank") == 3)
        .select(F.col("Academic_Year"), "FullDate")
    )

    # -------------------------------------------------------------------------
    # 3. Process for Year_Group information from dim_StudentAcademicYearGroup
    # -------------------------------------------------------------------------

    year_group_df = (
        group_df
        .filter(F.col("Group_Type") == 'YEAR')
        .select("groupkey", "Group_Description")
    )

    groupmembership_df = groupmembership_df.withColumnRenamed("studentkey", "gm_studentkey")
    year_group_df = year_group_df.withColumnRenamed("groupkey", "grp_groupkey")
    student_group_membership_df = (

        student_df
        .join(
            groupmembership_df,
            student_df["studentkey"] == groupmembership_df["gm_studentkey"],
            "inner"
        )

        .join(
            year_group_df,
            groupmembership_df["groupkey"] == year_group_df["grp_groupkey"],
            "inner"
        )

        .select(
            student_df["studentkey"],
            # Include school_id from dim_Student
            student_df["school_id"],
            # We'll also bring in "Sex" if we want from the same dim_Student
            student_df["Sex"],
            groupmembership_df["Start_Date"],
            #groupmembership_df["Membership_End_Date"],
            year_group_df["Group_Description"].alias("Year_Group")
        )

    )

    # -------------------------------------------------------------------------
    # 4. Academic Year Start/End => year_group_bridge_df
    # -------------------------------------------------------------------------

    student_group_membership_df = ensure_academic_year_column(student_group_membership_df, "Start_Date", school_id_column="school_id")

    from pyspark.sql import functions as F
    from pyspark.sql.window import Window

    # Define window spec
    window_spec = Window.partitionBy("studentkey", "Academic_Year").orderBy(F.desc("Start_Date"))

    # Add row_number and filter to keep only the first row
    year_group_bridge_df = (
        student_group_membership_df
        .withColumn("row_num", F.row_number().over(window_spec))
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )


    # -------------------------------------------------------------------------
    # 5. Student Extended categories => pivot
    # -------------------------------------------------------------------------

    student_extended_df = (
        student_extended_history_df
        .join(
            third_thursday_df,
            (

                (student_extended_history_df["Start_Date"] <= third_thursday_df["FullDate"]) &

                (
                    student_extended_history_df["End_Date"].isNull() |
                    (student_extended_history_df["End_Date"] >= third_thursday_df["FullDate"])
                )
            ),
            "inner"
        )
        .select(
            student_extended_history_df["studentkey"],
            third_thursday_df["Academic_Year"],
            student_extended_history_df["Student_Extended_Category"],
            F.lit(1).alias("Is_Active")
        )

    )

    pivot_df = (
        student_extended_df
        .groupBy("studentkey", "Academic_Year")
        .pivot("Student_Extended_Category")
        .agg(F.max("Is_Active"))
        .fillna(0)
    )

    final_df_pivot = (
        year_group_bridge_df
        .join(pivot_df, on=["studentkey", "Academic_Year"], how="full_outer")
    )

    cols_to_fill = [
        c for c in pivot_df.columns if c not in ["studentkey", "Academic_Year", "Year_Group", "school_id", "Sex"]
    ]

    final_df_no_nulls = final_df_pivot.fillna(0, subset=cols_to_fill)

    final_df_with_keycol = final_df_no_nulls.withColumn(
        "studentextendedacademicyearkey",
        F.monotonically_increasing_id()
    )

    # Filter only to records matching an Academic_Year + studentkey in saye_df
    final_df_match_aysk = (
        final_df_with_keycol
        .join(
            saye_df.select("Academic_Year", "studentkey").distinct(),
            on=["Academic_Year", "studentkey"],
            how="inner"
        )

    )

    # -------------------------------------------------------------------------
    # 6. Load & join dim_NCYearGroup => get std_Year_Group => then Cohort calc
    # -------------------------------------------------------------------------

    reference_path = get_reference_path(gold_path, "dim_NCYearGroup.csv")
    nc_year_group_df = spark.read.csv(reference_path, header=True, inferSchema=True)
    final_df_std_yg = final_df_match_aysk.join(
        nc_year_group_df.select(
            F.col("Year_Group_Description").alias("Year_Group_Join"),
            "std_Year_Group"
        ),
        final_df_match_aysk["Year_Group"] == F.col("Year_Group_Join"),
        "left"
    )

    # Basic cohort logic (as in your code)

    final_df_cohort = final_df_std_yg.withColumn(
        "Cohort",
        F.when(
            F.col("std_Year_Group") >= 11,
            F.col("Academic_Year")
        ).when(
            (F.col("std_Year_Group") >= 7) & (F.col("std_Year_Group") <= 10),
            F.expr("concat(split(Academic_Year, '/')[0] + (11 - std_Year_Group), '/', split(Academic_Year, '/')[1] + (11 - std_Year_Group))")
        ).when(
            F.col("std_Year_Group") < 6,
            F.expr("concat(split(Academic_Year, '/')[0] + (6 - std_Year_Group), '/', split(Academic_Year, '/')[1] + (6 - std_Year_Group))")
        )
    )

    final_df_no_dec = final_df_cohort.withColumn(
        "Cohort",
        F.concat(
            F.expr("CAST(split(Cohort, '/')[0] AS INT)").cast("string"),
            F.lit("/"),
            F.expr("CAST(split(Cohort, '/')[1] AS INT)").cast("string")
        )
    )

    final_df_KS = final_df_no_dec.withColumn(
        "Cohort",
        F.when(
            F.col("std_Year_Group") >= 11,
            F.concat(F.lit("KS4 "), F.col("Academic_Year"))
        ).when(
            (F.col("std_Year_Group") >= 7) & (F.col("std_Year_Group") <= 10),
            F.concat(
                F.lit("KS4 "),
                F.expr("CAST(split(Academic_Year, '/')[0] + (11 - std_Year_Group) AS INT)").cast("string"),
                F.lit("/"),
                F.expr("CAST(split(Academic_Year, '/')[1] + (11 - std_Year_Group) AS INT)").cast("string")
            )
        ).otherwise(
            F.concat(
                F.lit("KS2 "),
                F.expr("CAST(split(Academic_Year, '/')[0] + (6 - std_Year_Group) AS INT)").cast("string"),
                F.lit("/"),
                F.expr("CAST(split(Academic_Year, '/')[1] + (6 - std_Year_Group) AS INT)").cast("string")
            )
        )
    )

    # -------------------------------------------------------------------------
    # 7. Student full name (already in your code)
    #    Then JOIN the new tables for additional fields:
    #    - Is_EAL_String, Is_PP_String, Is_SEN_String from dim_StudentCalculated
    #    - Ethnicity, SEN_Status from dim_StudentExtended
    # -------------------------------------------------------------------------

    student_with_name_df = student_df.select(
        "studentkey",
        F.concat(
            F.col("Legal_Surname"),
            F.lit(", "),
            F.col("Legal_Forename")
        ).alias("Full_Name")

    )

    # 7A) For example, from dim_StudentCalculated => "Is_EAL_String", ...

    calc_fields_df = student_calculated_df.select(
        "studentkey",
        "Is_EAL_String"
    )

    # 7B) From dim_StudentExtended => "Ethnicity", "SEN_Status"

    extended_fields_df = student_extended_2_df.select(
        "studentkey",
        "Ethnicity"
    )

    final_df_step = final_df_KS.join(
        student_with_name_df, on="studentkey", how="left"

    ).join(
        calc_fields_df, on="studentkey", how="left"

    ).join(
        extended_fields_df, on="studentkey", how="left"

    )

    from pyspark.sql import functions as F

    # 7C Build additional string status fields with census data

    # Step 1: Rename the column and synchronise SEN columns with SEN assignments as the master
    final_df_step = final_df_step.withColumnRenamed("SEN Support", "SEN Support Plan")

    final_df_step = final_df_step.withColumn(
        "SEN_Status",
        F.when(
            (F.col("SEN Support Plan") == 1) | (F.col("Education, Health and Care Plan") == 1),
            1
        ).otherwise(0)
    )

    # Step 2: Add derived fields
    final_df_step = (
        final_df_step
        # Create Is_PP_String
        .withColumn(
            "Is_PP_String",
            F.when(F.col("Pupil_Premium_Indicator") == 1, F.lit("PP")).otherwise(F.lit("Not PP"))
        )
        # Create Is_SEN_String
        .withColumn(
            "Is_SEN_String",
            F.when(
                (F.col("Education, Health and Care Plan") == 1) | (F.col("SEN Support Plan") == 1),
                F.lit("SEN")
            ).otherwise(F.lit("Not SEN"))
        )
        # Create SEN_Support
        .withColumn(
            "SEN_Support",
            F.when(
                (F.col("Education, Health and Care Plan") == 1) & (F.col("SEN Support Plan") == 1),
                F.lit("E")
            ).when(
                F.col("Education, Health and Care Plan") == 1,
                F.lit("E")
            ).when(
                F.col("SEN Support Plan") == 1,
                F.lit("K")
            ).otherwise(F.lit(None))
        )
        # Create Child_Protection_Status
        .withColumn(
            "Child_Protection_Status",
            F.when(F.col("Child_Protection_Plan") == 1, F.lit("CP")).otherwise(F.lit("Not CP"))
        )
    )

    final_df_step = final_df_step.dropDuplicates(["studentextendedacademicyearkey"])

    # -------------------------------------------------------------------------
    # 8. Enrich with fields from sayg
    # -------------------------------------------------------------------------

    from pyspark.sql import functions as F

    # Read the sayg Parquet file back into a DataFrame
    sayg_df = spark.read.parquet(f"{gold_path}/dim_StudentAcademicYearGroup")

    # Keep only relevant columns
    sayg_filtered = sayg_df.select(
        "studentkey",
        "Academic_Year",
        "CompulsorySchoolAge",
        "OnRollOnCensus"
    )

    # Join with final_df_step on studentkey and Academic_Year
    final_df_enriched = final_df_step.join(
        sayg_filtered,
        on=["studentkey", "Academic_Year"],
        how="left"
    )

    # Apply the Academic_Year filter on the enriched DataFrame
    final_df_enriched = final_df_enriched.filter(
        F.col("Academic_Year").isin(academic_years)
    )

    # -------------------------------------------------------------------------
    # 9. Write out final
    # -------------------------------------------------------------------------


    final_df_enriched.write.mode("overwrite").parquet(f"{gold_path}/dim_StudentExtendedAcademicYear")

    # print("Successfully created/updated dim_StudentExtendedAcademicYear with added fields.")
    oeai.log.info("Successfully created/updated dim_StudentExtendedAcademicYear with added fields")
except Exception as e:
    # print(f"Error processing table {table_name}: {e}")
    oeai.log.exception(f"Error processing table {table_name}", table_name=table_name)

oeai.log.end_block()
oeai.log.checkpoint()

##### fact_AttendanceSummary

In [0]:
oeai.log.start_block("Create fact_AttendanceSummary")

table_name = "fact_AttendanceSummary"

try:

    from pyspark.sql.functions import sum as _sum, concat_ws, col, lit, when, collect_list, row_number, min, max, first, date_format, to_date
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType
    from pyspark.sql import Window

    fact_AttendanceSession = spark.read.format("delta").load(silver_path + 'fact_AttendanceSession')

    fact_AttendanceSession = ensure_academic_year_column(fact_AttendanceSession, "Date", school_id_column="school_id")

    # Cast boolean columns to integer before summing
    fact_AttendanceSession = fact_AttendanceSession.withColumn("is_present", col("is_present").cast("int")) \
                                                .withColumn("is_auth_abs", col("is_auth_abs").cast("int")) \
                                                .withColumn("is_unauth_abs", col("is_unauth_abs").cast("int")) \
                                                .withColumn("is_possible", col("is_possible").cast("int")) \
                                                .withColumn("is_attend", col("is_attend").cast("int")) \
                                                .withColumn("is_aea", col("is_aea").cast("int"))

    # Aggregating the fact_AttendanceSession dataframe by unique_student_id and AcademicYear
    aggregated_df = fact_AttendanceSession.groupBy("unique_student_id", "Academic_Year").agg(
        F.sum("is_present").alias("Present"),
        F.sum("is_auth_abs").alias("Authorised_Absences"),
        F.sum("is_unauth_abs").alias("Unauthorised_Absences"),
        F.sum("is_possible").alias("Possible_Marks"),
        F.sum("is_aea").alias("Approved_Educational_Activity"),
        F.sum(F.when(F.col("Mark") == "L", 1).otherwise(0)).alias("Late_Before_Registration"),
        F.sum(F.when(F.col("Mark") == "U", 1).otherwise(0)).alias("Late_After_Registration"),
        F.first("unique_key").alias("unique_key"),
        F.first("student_id").alias("student_id"),
        F.first("studentkey").alias("studentkey"),
        F.first("organisationkey").alias("organisationkey"),
        F.first("school_id").alias("school_id")
    )

    # Using the existing columns unique_key, student_id, and school_id from fact_AttendanceSession
    #aggregated_df = aggregated_df.withColumnRenamed("unique_key", "agg_unique_key") \
    #                             .withColumnRenamed("student_id", "agg_student_id") 
    #                             .withColumnRenamed("school_id", "agg_school_id")

    # Aggregation step 2: Concatenating Mark in date order
    window_spec = Window.partitionBy("unique_student_id", "Academic_Year").orderBy("Date", "Session")
    sorted_marks_df = fact_AttendanceSession.withColumn("sorted_mark", col("Mark")).withColumn("rn", row_number().over(window_spec))
    concatenated_marks_df = sorted_marks_df.groupBy("unique_student_id", "Academic_Year").agg(
        concat_ws("", collect_list("sorted_mark")).alias("Attendance_Mark_String")
    )

    concatenated_marks_df = concatenated_marks_df.withColumnRenamed("unique_student_id", "marks_unique_student_id")
    concatenated_marks_df = concatenated_marks_df.withColumnRenamed("Academic_Year", "marks_Academic_Year")

    # Joining aggregated data with concatenated marks
    summary_df = aggregated_df.join(concatenated_marks_df, 
                                    (aggregated_df["unique_student_id"] == concatenated_marks_df["marks_unique_student_id"]) & 
                                    (aggregated_df["Academic_Year"] == concatenated_marks_df["marks_Academic_Year"]),
                                    "left")

    summary_df = summary_df.drop(concatenated_marks_df["marks_unique_student_id"])
    summary_df = summary_df.drop(concatenated_marks_df["marks_Academic_Year"])


# Adding other necessary columns
    summary_df = summary_df.withColumn("Percentage_Attendance", ((col("Present") + col("Approved_Educational_Activity")) / col("Possible_Marks")).cast("decimal(10,4)")) \
                        .withColumn("Percentage_Authorised_Absence", (col("Authorised_Absences") / col("Possible_Marks")).cast("decimal(10,4)")) \
                        .withColumn("Percentage_Unauthorised_Absence", (col("Unauthorised_Absences") / col("Possible_Marks")).cast("decimal(10,4)")) \
                        .withColumn("external_id", lit(None).cast(StringType())) \
                        .withColumn("attendancesummarykey", lit(None).cast(StringType())) \
                        .withColumn("last_updated", lit(None).cast(StringType())) \
                        .withColumn("Unexplained_Absences", lit(None).cast(IntegerType())) \
                        .withColumn("Missing_Marks", lit(None).cast(IntegerType())) \
                        .withColumn("Percentage_Missing", lit(None).cast("decimal(10,4)")) \
                        .withColumn("Attendance_Not_Required", lit(None).cast(IntegerType())) \
                        .withColumn("Percentage_Unexplained_Absence", lit(None).cast(DecimalType(10, 0))) \
                        .withColumn(
                            "Is_Persistently_Absent",
                            when((col("Percentage_Attendance") <= 0.9) & (col("Possible_Marks") >= 20), 1).otherwise(0).cast(IntegerType())
                        ) \
                        .withColumn(
                            "Is_Severely_Absent",
                            when((col("Percentage_Attendance") <= 0.5) & (col("Possible_Marks") >= 20), 1).otherwise(0).cast(IntegerType())
                        )

    #  attendance bands
    df_attendance_summary = summary_df
    df_attendance_summary = df_attendance_summary.withColumn("under_50",when(col("Percentage_Attendance") <= 0.5, 1).otherwise(0)) 
    df_attendance_summary = df_attendance_summary.withColumn("50_to_70",when((col("Percentage_Attendance") > 0.5) & (col("Percentage_Attendance") <= 0.7), 1).otherwise(0))
    df_attendance_summary = df_attendance_summary.withColumn("70_to_80",when((col("Percentage_Attendance") > 0.7) & (col("Percentage_Attendance") <= 0.8), 1).otherwise(0))
    df_attendance_summary = df_attendance_summary.withColumn("80_to_90",when((col("Percentage_Attendance") > 0.8) & (col("Percentage_Attendance") <= 0.9), 1).otherwise(0))
    df_attendance_summary = df_attendance_summary.withColumn("90_to_92",when((col("Percentage_Attendance") > 0.9) & (col("Percentage_Attendance") <= 0.92), 1).otherwise(0))
    df_attendance_summary = df_attendance_summary.withColumn("92_to_95",when((col("Percentage_Attendance") > 0.92) & (col("Percentage_Attendance") <= 0.95), 1).otherwise(0))
    df_attendance_summary = df_attendance_summary.withColumn("95_to_98",when((col("Percentage_Attendance") > 0.95) & (col("Percentage_Attendance") <= 0.98), 1).otherwise(0))
    df_attendance_summary = df_attendance_summary.withColumn("above_98",when(col("Percentage_Attendance") > 0.98, 1).otherwise(0))             

    # Create a single 'Attendance_Bin' column
    df_attendance_summary = df_attendance_summary.withColumn(
        "Attendance_Bin",
        when(col("Percentage_Attendance") <= 0.5, "under 50%")
        .when((col("Percentage_Attendance") > 0.5) & (col("Percentage_Attendance") <= 0.7), "50% to 70%")
        .when((col("Percentage_Attendance") > 0.7) & (col("Percentage_Attendance") <= 0.8), "70% to 80%")
        .when((col("Percentage_Attendance") > 0.8) & (col("Percentage_Attendance") <= 0.9), "80% to 90%")
        .when((col("Percentage_Attendance") > 0.9) & (col("Percentage_Attendance") <= 0.92), "90% to 92%")
        .when((col("Percentage_Attendance") > 0.92) & (col("Percentage_Attendance") <= 0.95), "92% to 95%")
        .when((col("Percentage_Attendance") > 0.95) & (col("Percentage_Attendance") <= 0.98), "95% to 98%")
        .when(col("Percentage_Attendance") > 0.98, "above 98%")
        .otherwise("Unspecified")  # This is optional and can handle any data outside the expected ranges
    )

    # Populate fact_AttendanceSummary
    fact_AttendanceSummary = df_attendance_summary

    #  add the unique_key column to the AttendanceSession dataframe
    fact_AttendanceSummary = fact_AttendanceSummary.withColumn("unique_key", concat(col("unique_student_id"), col("Academic_Year")))

    deltaTablePath_AttSum = silver_path + "fact_AttendanceSummary"
    fact_AttendanceSummary.write.format("delta").mode("overwrite").save(deltaTablePath_AttSum)
    oeai.log.info(f"Table {table_name} processed successfully.")
except Exception as e:
    # print(f"Error processing table {table_name}: {e}")
    oeai.log.exception(f"Error processing table {table_name}", table_name=table_name)

oeai.log.end_block()
oeai.log.checkpoint()


##### Main processing block.  Iterates through each delta table to optimise, enrich and produce the parquet in the Gold layer.

In [0]:
oeai.log.start_block("Write to Gold Layer")

from datetime import datetime
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.functions import col
from pyspark.sql import Window
from pyspark.sql.functions import desc, row_number

# Fact tables processing loop with debugging
for table_name in delta_tables:
    silver_path_delta = f"{silver_path}/{table_name}"
    gold_path_delta = f"{gold_path}/{table_name}"
    
    try:
        delta_table = DeltaTable.forPath(spark, silver_path_delta)
        df = delta_table.toDF()
        initial_count = df.count()
        
        # Apply transformations based on table
        if table_name == "fact_AttendanceSession":
            # --------------------------------------------------------------------------
            # 1) Rename, ensure schema adjustments, filter
            # --------------------------------------------------------------------------
            df = df.drop("external_id", "unique_student_id", "student_id", "attendancesessionkey",
                         "Comment", "Employee", "last_updated", "unique_key", "organisationkey")
            df = ensure_academic_year_column(df, "Date", school_id_column="school_id")
            df = add_student_academic_year_group_key(df)
            try:
                df = add_student_ext_academic_year_key(df)
            except Exception as e:
                    # print(f"Error adding student_ext_academic_year_key {e}")
                    oeai.log.exception(f"Error adding student_ext_academic_year_key", table_name=table_name) 

            df = df.filter(col("studentacademicyeargroupkey").cast("int").isNotNull())

            # Cast to int
            df = (df
                .withColumn("is_present", F.col("is_present").cast("int"))
                .withColumn("is_auth_abs", F.col("is_auth_abs").cast("int"))
                .withColumn("is_unauth_abs", F.col("is_unauth_abs").cast("int"))
                .withColumn("is_possible", F.col("is_possible").cast("int"))
                .withColumn("is_aea", F.col("is_aea").cast("int"))
                .withColumn("is_attend", F.col("is_attend").cast("int"))
                .withColumn("is_nr", F.col("is_nr").cast("int"))
                .withColumn("is_late_L", F.col("is_late_L").cast("int"))
                .withColumn("is_late_U", F.col("is_late_U").cast("int"))
                .withColumn("is_missing", F.col("is_missing").cast("int"))
            )   
        
            # --------------------------------------------------------------------------
            # 2) Convert Date to a proper date type for window ops
            # --------------------------------------------------------------------------
            df = df.withColumn("Date_as_date", F.to_date(F.col("Date"), "dd/MM/yyyy"))  
        
            # --------------------------------------------------------------------------
            # 3) Rolling 10-week logic for 'Is_10_in_10'
            #    - For each session row, sum is_unauth_abs across the last 10 *active* weeks
            #    - A week is "active" if it has one or more 'is_possible' marks for that student
            #    - If that sum >= 10 at this row, flag = 1
            # --------------------------------------------------------------------------
            # 3a) Define a "week_start" column to group sessions into weeks
            df = df.withColumn("week_start", F.date_trunc("week", F.col("Date_as_date")))
        
            # 3b) Summarise unauthorised absences and possible marks at week level
            week_df = (
                df.groupBy("studentacademicyeargroupkey", "Academic_Year", "week_start")
                .agg(
                    F.sum("is_unauth_abs").alias("unauth_abs_per_week"),
                    F.sum("is_possible").alias("possible_per_week")
                )
                # Keep only active weeks
                .filter(F.col("possible_per_week") > 0)
            )
        
            # 3c) Define a rolling window of the last 10 active weeks
            week_window_10 = (
                Window
                .partitionBy("studentacademicyeargroupkey", "Academic_Year")
                .orderBy("week_start")
                .rowsBetween(-9, 0)  # includes current week + 9 preceding active weeks
            )
        
            # 3d) Rolling sum of unauthorised absences across those 10 active weeks
            week_df = week_df.withColumn(
                "rolling_10week_unauth_abs",
                F.sum("unauth_abs_per_week").over(week_window_10)
            )
        
            # 3e) Flag: if rolling_10week_unauth_abs >= 10 => Is_10_in_10_week = 1
            week_df = week_df.withColumn(
                "Is_10_in_10_week",
                F.when(F.col("rolling_10week_unauth_abs") >= 10, 1).otherwise(0)
            )
        
            # 3f) Bring the week-level flag back to session-level
            df = (
                df.alias("sess")
                .join(
                    week_df.select(
                        "studentacademicyeargroupkey", 
                        "Academic_Year", 
                        "week_start", 
                        "Is_10_in_10_week"
                    ).alias("wk"),
                    on=["studentacademicyeargroupkey", "Academic_Year", "week_start"],
                    how="left"
                )
                .withColumn("Is_10_in_10", F.col("wk.Is_10_in_10_week"))
                .drop("wk.Is_10_in_10_week")
            )
            # 3g) Fill forward the last known Is_10_in_10 value across sessions
            fill_window = (
                Window
                .partitionBy("studentacademicyeargroupkey", "Academic_Year")
                .orderBy("Date_as_date", "Session")
                .rowsBetween(Window.unboundedPreceding, 0)
            )

            df = df.withColumn(
                "Is_10_in_10_filled",
                F.last("Is_10_in_10", ignorenulls=True).over(fill_window)
            ).drop("Is_10_in_10") \
            .withColumnRenamed("Is_10_in_10_filled", "Is_10_in_10")
        
            # --------------------------------------------------------------------------
            # 4) 3-day consecutive absence -> '3_days_abs' = 1 if this session's day
            #    is part of at least 3 consecutive absent days.
            # --------------------------------------------------------------------------
            daily_abs_df = (df
                .groupBy("studentacademicyeargroupkey", "Academic_Year", "Date_as_date")
                .agg(F.sum(F.col("is_auth_abs") + F.col("is_unauth_abs")).alias("absences_that_day"))
                .withColumn("is_abs_day", F.when(F.col("absences_that_day") > 0, 1).otherwise(0))
            )
        
            day_window = (
                Window
                .partitionBy("studentacademicyeargroupkey", "Academic_Year")
                .orderBy("Date_as_date")
            )
            daily_abs_df = (daily_abs_df
                .withColumn("prev1", F.lag("is_abs_day", 1).over(day_window))
                .withColumn("prev2", F.lag("is_abs_day", 2).over(day_window))
                .withColumn(
                    "has_3consecutive",
                    F.when(
                        (F.col("is_abs_day") == 1) &
                        (F.col("prev1") == 1) &
                        (F.col("prev2") == 1),
                        1
                    ).otherwise(0)
                )
            )
        
            daily_abs_df = daily_abs_df.select(
                "studentacademicyeargroupkey", "Academic_Year", "Date_as_date", "has_3consecutive"
            )
            df = (df
                .join(
                    daily_abs_df,
                    on=["studentacademicyeargroupkey", "Academic_Year", "Date_as_date"],
                    how="left"
                )
                .withColumn("3_days_abs", F.col("has_3consecutive"))
                .drop("has_3consecutive")
            )
        
            # --------------------------------------------------------------------------
            # 5) abs_after_exclusion -> if the PREVIOUS session Mark == 'E' and
            #    CURRENT session Mark in [C,E,G,H,I,M,N,O,R,S,T,U,C1,C2,J1]
            # --------------------------------------------------------------------------
            absence_codes = ["C","G","H","I","M","N","O","R","S","T","U","C1","C2","J1"]
        
            session_window_unbounded = (
                Window
                .partitionBy("studentacademicyeargroupkey", "Academic_Year")
                .orderBy("Date_as_date", "Session")
                .rowsBetween(Window.unboundedPreceding, 0)
            )
        
            df = df.withColumn(
                "exclusion_count",
                F.sum(F.when(F.col("Mark") == "E", 1).otherwise(0)).over(session_window_unbounded)
            )
        
            df = df.withColumn(
                "non_absence_count",
                F.sum(
                    F.when(
                        ~F.col("Mark").isin(absence_codes + ["E"]), 1
                    ).otherwise(0)
                ).over(session_window_unbounded)
            )
        
            df = df.withColumn(
                "abs_after_exclusion",
                F.when(
                    (F.col("exclusion_count") > F.col("non_absence_count"))
            & (F.col("Mark").isin(absence_codes)),
                    1
                ).otherwise(0)
            )
        
            df = df.drop("exclusion_count", "non_absence_count")
            # --------------------------------------------------------------------------
            # 6) Define a Window to accumulate from the start of each academic year
            # --------------------------------------------------------------------------
            window_spec = (
                Window
                .partitionBy("studentacademicyeargroupkey", "Academic_Year")
                .orderBy(F.col("Date").asc())
                .rowsBetween(Window.unboundedPreceding, Window.currentRow)
            )
            # --------------------------------------------------------------------------
            # 7) Compute cumulative sums, derive attendance ratio
            # --------------------------------------------------------------------------
            df = (df
                .withColumn("cumulative_present", F.sum("is_attend").over(window_spec))
                .withColumn("cumulative_possible", F.sum("is_possible").over(window_spec))
                .withColumn(
                    "attendance_ratio",
                    F.when(
                        F.col("cumulative_possible") > 0,
                        F.col("cumulative_present") / F.col("cumulative_possible")
                    )
                )
            )
        
            # --------------------------------------------------------------------------
            # 8) Label each row with is_PA / is_SA
            #    - is_PA = 1 if attendance_ratio <= 0.90 and sum_possible > 20
            #    - is_SA = 1 if attendance_ratio <= 0.50 and sum_possible > 20
            # --------------------------------------------------------------------------
            df = (df
                .withColumn(
                    "is_PA",
                    F.when(
                        (F.col("attendance_ratio") <= 0.90) & (F.col("cumulative_possible") > 20),
                        1
                    ).otherwise(0)
                )
                .withColumn(
                    "is_SA",
                    F.when(
                        (F.col("attendance_ratio") <= 0.50) & (F.col("cumulative_possible") > 20),
                        1
                    ).otherwise(0)
                )
            )
            # --------------------------------------------------------------------------
            # Final cleanup
            # --------------------------------------------------------------------------
            df = df.drop("Date_as_date", "week_start", "rolling_10week_unauth_abs", "Is_10_in_10_week")

        elif table_name == "fact_Behaviour":
            df = df.drop("behaviourkey", "external_id", "unique_student_id", "unique_key", "student_id", 
                         "organisationkey", "Location", "Status", "Comment", "Subject", "Class", "Total_Points", "last_updated", "Is_Deleted")
            df = df.withColumn("Points", F.abs(F.col("Points")))  # Make Points absolute
            df = ensure_academic_year_column(df, "Incident_Date", school_id_column="school_id")
            df = add_student_academic_year_group_key(df)
            try:
                df = add_student_ext_academic_year_key(df)
            except Exception as e:
                # print(f"Error adding student_ext_academic_year_key {e}") 
                oeai.log.exception(f"Error adding student_ext_academic_year_key", table_name=table_name) 

            df = df.filter(col("studentacademicyeargroupkey").cast("int").isNotNull())


        elif table_name == "fact_Achievement":
            df = df.drop("achievementkey", "external_id", "organisationkey", "unique_student_id", "unique_key",
                         "student_id", "action_date", "Subject", "Class", "Total_Points", "Comment", "Parents_Notified", "last_updated", "Is_Deleted")
            df = df.withColumn("Points", F.abs(F.col("Points")))  # Make Points absolute
            df = ensure_academic_year_column(df, "Achievement_Date", school_id_column="school_id")
            df = add_student_academic_year_group_key(df)
            try:
                df = add_student_ext_academic_year_key(df)
            except Exception as e:
                # print(f"Error adding student_ext_academic_year_key {e}")
                oeai.log.exception(f"Error adding student_ext_academic_year_key", table_name=table_name)
            df = df.filter(col("studentacademicyeargroupkey").cast("int").isNotNull())
        
        elif table_name == "fact_CourseEnrolment":
            df = ensure_academic_year_column(df, "Start_Date", school_id_column="school_id")
            df = add_student_academic_year_group_key(df)
            try:
                df = add_student_ext_academic_year_key(df)
            except Exception as e:
                # print(f"Error adding student_ext_academic_year_key {e}") 
                oeai.log.exception(f"Error adding student_ext_academic_year_key", table_name=table_name)

        elif table_name == "fact_AttendanceSummary":
            df = add_student_academic_year_group_key(df)
            try:
                df = add_student_ext_academic_year_key(df)
            except Exception as e:
                # print(f"Error adding student_ext_academic_year_key {e}") 
                oeai.log.exception(f"Error adding student_ext_academic_year_key", table_name=table_name)
        
        elif table_name == "fact_AttendanceLesson":
            df = df.filter(col("Subject").isNotNull())
            df = ensure_academic_year_column(df, "Start", school_id_column="school_id")
            df = add_student_academic_year_group_key(df)
            try:
                df = add_student_ext_academic_year_key(df)
            except Exception as e:
                # print(f"Error adding student_ext_academic_year_key {e}") 
                oeai.log.exception(f"Error adding student_ext_academic_year_key", table_name=table_name)

        elif table_name == "dim_StudentExtended":
            df = df.withColumn("Is_Current", when(col("Is_Current") == "true", "1").otherwise("0"))
        
        elif table_name == "fact_ExamResult":
            df = add_student_academic_year_group_key(df)
            try:
                df = add_student_ext_academic_year_key(df)
            except Exception as e:
                # print(f"Error adding student_ext_academic_year_key {e}") 
                oeai.log.exception(f"Error adding student_ext_academic_year_key", table_name=table_name)
            df = df.filter(col("studentacademicyeargroupkey").cast("int").isNotNull())
        
        elif table_name == "fact_Detention":
            df = ensure_academic_year_column(df, "Start", school_id_column="school_id")
            df = add_student_academic_year_group_key(df)
            try:
                df = add_student_ext_academic_year_key(df)
            except Exception as e:
                # print(f"Error adding student_ext_academic_year_key {e}") 
                oeai.log.exception(f"Error adding student_ext_academic_year_key", table_name=table_name)

        elif table_name == "fact_PointAward":
            df = ensure_academic_year_column(df, "Date", school_id_column="school_id")
            df = add_student_academic_year_group_key(df)
            try:
                df = add_student_ext_academic_year_key(df)
            except Exception as e:
                # print(f"Error adding student_ext_academic_year_key {e}") 
                oeai.log.exception(f"Error adding student_ext_academic_year_key", table_name=table_name)
        
        elif table_name == "fact_InternalExclusion":
            df = ensure_academic_year_column(df, "Start", school_id_column="school_id")
            df = add_student_academic_year_group_key(df)
            try:
                df = add_student_ext_academic_year_key(df)
            except Exception as e:
                # print(f"Error adding student_ext_academic_year_key {e}") 
                oeai.log.exception(f"Error adding student_ext_academic_year_key", table_name=table_name)

        elif table_name == "fact_Exclusion":
            df = df.drop("Academic_Year")

            df = ensure_academic_year_column(df, "Start_Date", school_id_column="school_id")

            df = add_student_academic_year_group_key(df)
            try:
                df = add_student_ext_academic_year_key(df)
            except Exception as e:
                # print(f"Error adding student_ext_academic_year_key {e}") 
                oeai.log.exception(f"Error adding student_ext_academic_year_key", table_name=table_name)

            df = df.filter(col("studentacademicyeargroupkey").cast("int").isNotNull())
            
            df = df.withColumn("Days", F.when((F.col("Days").isNull()) | (F.col("Days") == "None"), 0).otherwise(F.col("Days")))
            
            # Standardize Exclusion Types
            df = df.withColumn("Type", 
                               F.when(F.col("Type") == "FixedPeriodExclusion", "Suspension")
                                .when(F.col("Type") == "PermanentExclusion", "Permanent")
                                .otherwise(F.col("Type")))
            df = df.withColumn("Type_Code", 
                               F.when(F.col("Type_Code") == "FixedPeriodExclusion", "SUSP")
                                .when(F.col("Type_Code") == "PermanentExclusion", "PERM")
                                .otherwise(F.col("Type_Code")))

            # Drop duplicates based on the specified columns to handle the situation where a school deletes an exclusion and replaces it resulting in a duplicate
            df = df.dropDuplicates(['Academic_Year', 'school_id', 'student_id', 'Start_Date', 'End_Date', 'Days'])

        # Write the DataFrame to Parquet
        # print(f"Table: {table_name} now writing out")
        df.write.format("parquet").mode("overwrite").save(gold_path_delta)
        oeai.log.info(f"Written out table: {table_name}", table_name=table_name)
    except Exception as e:
        print(f"Error processing table {table_name}: {e}")
        oeai.log.exception(f"Error processing table {table_name}", table_name=table_name)

oeai.log.end_block()
oeai.log.checkpoint()

In [0]:
oeai.log.end_block(index=0, include_target=True)
oeai.log.checkpoint("Write gold tables")

##### Create fact_AttendanceTrends

In [0]:
oeai.log.start_block("Create fact_AttendanceTrends")

In [0]:
# =========================================================
# Attendance Trends (multi-AY rows) with SAFE end dates
# Current, Prior, Second-Prior; Second-Prior compares to Third-Prior
# =========================================================

try:
    import os
    from datetime import date, datetime, timedelta

    from pyspark.sql import functions as F
    from pyspark.sql.window import Window

    # Safety margin for AY end dates (prevents overlaps)
    SAFE_END_DAYS = 14

    # ---------------------------
    # Helpers
    # ---------------------------
    def df_is_empty(df) -> bool:
        try:
            return len(df.take(1)) == 0
        except Exception:
            return True

    def prior_n_year_str(ay_str: str, n: int = 1):
        """
        Return the academic year string n years before ay_str.
        Example: "2024/25", n=2 -> "2022/23"
        """
        import re
        m = re.fullmatch(r"\s*(\d{4})\s*/\s*(\d{4})\s*", ay_str or "")
        if not m:
            return None
        y1, y2 = int(m.group(1)), int(m.group(2))
        return f"{y1 - n}/{y2 - n}"

    def infer_from_today(today: date | None = None, aug_start_month: int = 8, aug_start_day: int = 1):
        """
        Fallback “academic year” calculator with 1 Aug start.
        Returns current + prior + second-prior + third-prior windows.
        """
        if today is None:
            today = date.today()

        start_year = today.year if (today.month, today.day) >= (aug_start_month, aug_start_day) else (today.year - 1)
        end_year = start_year + 1

        ay_start = date(start_year, aug_start_month, aug_start_day)
        ay_end_full = date(end_year, 7, 31)
        ay_end   = ay_end_full - timedelta(days=1)

        prior1_start = date(start_year - 1, aug_start_month, aug_start_day)
        prior1_end   = ay_start - timedelta(days=1)

        prior2_start = date(start_year - 2, aug_start_month, aug_start_day)
        prior2_end   = prior1_start - timedelta(days=1)

        prior3_start = date(start_year - 3, aug_start_month, aug_start_day)
        prior3_end   = prior2_start - timedelta(days=1)

        return {
            "CURRENT_ACADEMIC_YEAR": f"{start_year}/{end_year}",
            "ACADEMIC_YEAR_START":   ay_start.strftime("%Y-%m-%d"),
            "ACADEMIC_YEAR_END":     ay_end.strftime("%Y-%m-%d"),
            "CURRENT_SCHOOL_YEAR_START": ay_start.strftime("%Y-%m-%d"),

            "PRIOR_ACADEMIC_YEAR":   f"{start_year - 1}/{end_year - 1}",
            "PRIOR_SCHOOL_YEAR_START": prior1_start.strftime("%Y-%m-%d"),
            "PRIOR_SCHOOL_YEAR_END":   prior1_end.strftime("%Y-%m-%d"),

            "SECOND_PRIOR_ACADEMIC_YEAR": f"{start_year - 2}/{end_year - 2}",
            "SECOND_PRIOR_SCHOOL_YEAR_START": prior2_start.strftime("%Y-%m-%d"),
            "SECOND_PRIOR_SCHOOL_YEAR_END":   prior2_end.strftime("%Y-%m-%d"),

            "THIRD_PRIOR_ACADEMIC_YEAR": f"{start_year - 3}/{end_year - 3}",
            "THIRD_PRIOR_SCHOOL_YEAR_START": prior3_start.strftime("%Y-%m-%d"),
            "THIRD_PRIOR_SCHOOL_YEAR_END":   prior3_end.strftime("%Y-%m-%d"),
        }

    # ---------------------------
    # Load sources
    # ---------------------------
    assert 'spark' in globals(), "spark session is not defined"
    assert 'gold_path' in globals(), "gold_path is not defined"
    assert 'silver_path' in globals(), "silver_path is not defined"

    # Sessions
    sessions_path = os.path.join(gold_path, "fact_AttendanceSession")
    df_sessions = spark.read.parquet(sessions_path)
    # Remove weekends and holidays if MIS returns them
    df_sessions = df_sessions.filter(df_sessions.Mark != '#')
    
    print(f"Loaded sessions from Gold (Parquet): {df_sessions.count():,} rows")

    # Student dim
    df_dim_student = spark.read.format("delta").load(f"{silver_path}/dim_Student")
    df_dim_student_extended = spark.read.format("delta").load(f"{silver_path}/dim_StudentExtended")
    df_dim_organisation = spark.read.format("delta").load(f"{silver_path}/dim_Organisation")

    df_student_master = (
        df_dim_student.select(
            F.col("studentkey"),
            F.col("Forename"),
            F.col("Surname"),
            F.concat_ws(", ", F.col("Surname"), F.col("Forename")).alias("Student Name"),
            F.col("Sex"),
            F.col("UPN"),
            F.col("organisationkey")
        )
        .join(
            df_dim_student_extended.select(
                F.col("studentkey"),
                F.col("Pupil_Premium_Indicator").alias("Pupil Premium Indicator"),
                F.col("SEN_Status").alias("SEND Status"),
                F.col("Enrolment_Status").alias("Enrolment Status"),
                F.col("Year_Group").alias("Year Group"),
            ),
            on="studentkey",
            how="left"
        )
        .join(
            df_dim_organisation.select(
                F.col("organisationkey"),
                F.col("Organisation_name").alias("School Name")
            ),
            on="organisationkey",
            how="left"
        )
        .drop("organisationkey")
        .cache()
    )
    print(f"✓ Master student dimension ready: {df_student_master.count():,} students")

    # ---------------------------
    # Academic Year resolution (table -> fallback) with SAFE ends
    # ---------------------------
    df_years = None
    used_fallback = False
    try:
        dim_years_path = silver_path + "dim_AcademicYear"
        df_years = spark.read.format("delta").load(dim_years_path)
    except Exception:
        used_fallback = True

    if not used_fallback:
        needed = {"ACADEMIC_YEAR_NAME", "CALENDAR_YEAR_START_DATE", "CALENDAR_YEAR_END_DATE"}
        if df_is_empty(df_years) or not needed.issubset(set(df_years.columns)):
            used_fallback = True
        else:
            today_col = F.current_date()
            df_years = df_years.withColumn(
                "IS_CURRENT",
                F.when(
                    (F.to_date(F.col("CALENDAR_YEAR_START_DATE")) <= today_col) &
                    (today_col <= F.to_date(F.col("CALENDAR_YEAR_END_DATE"))),
                    F.lit(True)
                ).otherwise(F.lit(False))
            )

    CURRENT_ACADEMIC_YEAR = ACADEMIC_YEAR_START = ACADEMIC_YEAR_END = CURRENT_SCHOOL_YEAR_START = None
    PRIOR_ACADEMIC_YEAR = PRIOR_SCHOOL_YEAR_START = PRIOR_SCHOOL_YEAR_END = None
    SECOND_PRIOR_ACADEMIC_YEAR = SECOND_PRIOR_SCHOOL_YEAR_START = SECOND_PRIOR_SCHOOL_YEAR_END = None
    THIRD_PRIOR_ACADEMIC_YEAR  = THIRD_PRIOR_SCHOOL_YEAR_START  = THIRD_PRIOR_SCHOOL_YEAR_END  = None

    if not used_fallback:
        current_df = df_years.filter(F.col("IS_CURRENT") == True)
        if df_is_empty(current_df):
            used_fallback = True
        else:
            # Current AY — start from table, end = max(end) - SAFE_END_DAYS (but never before start)
            cur_agg = current_df.agg(
                F.min(F.to_date("CALENDAR_YEAR_START_DATE")).alias("start"),
                F.greatest(
                    F.min(F.to_date("CALENDAR_YEAR_START_DATE")),
                    F.date_sub(F.max(F.to_date("CALENDAR_YEAR_END_DATE")), SAFE_END_DAYS)
                ).alias("end_safe")
            ).first()
            ACADEMIC_YEAR_START = str(cur_agg["start"]) if cur_agg else None
            ACADEMIC_YEAR_END   = str(cur_agg["end_safe"]) if cur_agg else None

            first_row = current_df.select("ACADEMIC_YEAR_NAME").limit(1).collect()
            CURRENT_ACADEMIC_YEAR = str(first_row[0]["ACADEMIC_YEAR_NAME"]) if first_row else None
            CURRENT_SCHOOL_YEAR_START = ACADEMIC_YEAR_START

            # Prior (−1)
            PRIOR_ACADEMIC_YEAR = prior_n_year_str(CURRENT_ACADEMIC_YEAR, 1)
            prior1_df = df_years.filter(F.col("ACADEMIC_YEAR_NAME") == PRIOR_ACADEMIC_YEAR)
            if not df_is_empty(prior1_df):
                p1 = prior1_df.agg(
                    F.min(F.to_date("CALENDAR_YEAR_START_DATE")).alias("start"),
                    F.greatest(
                        F.min(F.to_date("CALENDAR_YEAR_START_DATE")),
                        F.date_sub(F.max(F.to_date("CALENDAR_YEAR_END_DATE")), SAFE_END_DAYS)
                    ).alias("end_safe")
                ).first()
                PRIOR_SCHOOL_YEAR_START = str(p1["start"]) if p1 else None
                PRIOR_SCHOOL_YEAR_END   = str(p1["end_safe"]) if p1 else None
            else:
                try:
                    csys = datetime.strptime(CURRENT_SCHOOL_YEAR_START, "%Y-%m-%d").date()
                    PRIOR_SCHOOL_YEAR_START = csys.replace(year=csys.year - 1).strftime("%Y-%m-%d")
                    PRIOR_SCHOOL_YEAR_END   = (csys - timedelta(days=SAFE_END_DAYS)).strftime("%Y-%m-%d")
                except Exception:
                    pass

            # Second Prior (−2)
            SECOND_PRIOR_ACADEMIC_YEAR = prior_n_year_str(CURRENT_ACADEMIC_YEAR, 2)
            prior2_df = df_years.filter(F.col("ACADEMIC_YEAR_NAME") == SECOND_PRIOR_ACADEMIC_YEAR)
            if not df_is_empty(prior2_df):
                p2 = prior2_df.agg(
                    F.min(F.to_date("CALENDAR_YEAR_START_DATE")).alias("start"),
                    F.greatest(
                        F.min(F.to_date("CALENDAR_YEAR_START_DATE")),
                        F.date_sub(F.max(F.to_date("CALENDAR_YEAR_END_DATE")), SAFE_END_DAYS)
                    ).alias("end_safe")
                ).first()
                SECOND_PRIOR_SCHOOL_YEAR_START = str(p2["start"]) if p2 else None
                SECOND_PRIOR_SCHOOL_YEAR_END   = str(p2["end_safe"]) if p2 else None
            else:
                try:
                    p1s = datetime.strptime(PRIOR_SCHOOL_YEAR_START, "%Y-%m-%d").date()
                    SECOND_PRIOR_SCHOOL_YEAR_START = p1s.replace(year=p1s.year - 1).strftime("%Y-%m-%d")
                    SECOND_PRIOR_SCHOOL_YEAR_END   = (p1s - timedelta(days=SAFE_END_DAYS)).strftime("%Y-%m-%d")
                except Exception:
                    pass

            # Third Prior (−3) — used as comparison for Second-Prior row
            THIRD_PRIOR_ACADEMIC_YEAR = prior_n_year_str(CURRENT_ACADEMIC_YEAR, 3)
            prior3_df = df_years.filter(F.col("ACADEMIC_YEAR_NAME") == THIRD_PRIOR_ACADEMIC_YEAR)
            if not df_is_empty(prior3_df):
                p3 = prior3_df.agg(
                    F.min(F.to_date("CALENDAR_YEAR_START_DATE")).alias("start"),
                    F.greatest(
                        F.min(F.to_date("CALENDAR_YEAR_START_DATE")),
                        F.date_sub(F.max(F.to_date("CALENDAR_YEAR_END_DATE")), SAFE_END_DAYS)
                    ).alias("end_safe")
                ).first()
                THIRD_PRIOR_SCHOOL_YEAR_START = str(p3["start"]) if p3 else None
                THIRD_PRIOR_SCHOOL_YEAR_END   = str(p3["end_safe"]) if p3 else None
            else:
                try:
                    p2s = datetime.strptime(SECOND_PRIOR_SCHOOL_YEAR_START, "%Y-%m-%d").date()
                    THIRD_PRIOR_SCHOOL_YEAR_START = p2s.replace(year=p2s.year - 1).strftime("%Y-%m-%d")
                    THIRD_PRIOR_SCHOOL_YEAR_END   = (p2s - timedelta(days=SAFE_END_DAYS)).strftime("%Y-%m-%d")
                except Exception:
                    pass

    if used_fallback:
        inferred = infer_from_today(today=date.today(), aug_start_month=8, aug_start_day=1)
        CURRENT_ACADEMIC_YEAR          = inferred["CURRENT_ACADEMIC_YEAR"]
        ACADEMIC_YEAR_START            = inferred["ACADEMIC_YEAR_START"]
        ACADEMIC_YEAR_END              = inferred["ACADEMIC_YEAR_END"]
        CURRENT_SCHOOL_YEAR_START      = inferred["CURRENT_SCHOOL_YEAR_START"]
        PRIOR_ACADEMIC_YEAR            = inferred["PRIOR_ACADEMIC_YEAR"]
        PRIOR_SCHOOL_YEAR_START        = inferred["PRIOR_SCHOOL_YEAR_START"]
        PRIOR_SCHOOL_YEAR_END          = inferred["PRIOR_SCHOOL_YEAR_END"]
        SECOND_PRIOR_ACADEMIC_YEAR     = inferred["SECOND_PRIOR_ACADEMIC_YEAR"]
        SECOND_PRIOR_SCHOOL_YEAR_START = inferred["SECOND_PRIOR_SCHOOL_YEAR_START"]
        SECOND_PRIOR_SCHOOL_YEAR_END   = inferred["SECOND_PRIOR_SCHOOL_YEAR_END"]
        THIRD_PRIOR_ACADEMIC_YEAR      = inferred["THIRD_PRIOR_ACADEMIC_YEAR"]
        THIRD_PRIOR_SCHOOL_YEAR_START  = inferred["THIRD_PRIOR_SCHOOL_YEAR_START"]
        THIRD_PRIOR_SCHOOL_YEAR_END    = inferred["THIRD_PRIOR_SCHOOL_YEAR_END"]
        print("INFO: Academic year derived from today using 1 Aug start convention (SAFE ends).")

    # "today-equivalents" for YTD caps
    TODAY = datetime.now().strftime("%Y-%m-%d")
    PRIOR_YEAR_TODAY  = (datetime.now() - timedelta(days=365)).strftime("%Y-%m-%d")
    PRIOR2_YEAR_TODAY = (datetime.now() - timedelta(days=730)).strftime("%Y-%m-%d")
    PRIOR3_YEAR_TODAY = (datetime.now() - timedelta(days=1095)).strftime("%Y-%m-%d")

    print(f"\nDate Configuration:")
    print(f"  Current AY: {CURRENT_ACADEMIC_YEAR} ({ACADEMIC_YEAR_START} → {ACADEMIC_YEAR_END})")
    print(f"  Prior AY: {PRIOR_ACADEMIC_YEAR} ({PRIOR_SCHOOL_YEAR_START} → {PRIOR_SCHOOL_YEAR_END})")
    print(f"  2-Years-Prior AY: {SECOND_PRIOR_ACADEMIC_YEAR} ({SECOND_PRIOR_SCHOOL_YEAR_START} → {SECOND_PRIOR_SCHOOL_YEAR_END})")
    print(f"  3-Years-Prior AY: {THIRD_PRIOR_ACADEMIC_YEAR} ({THIRD_PRIOR_SCHOOL_YEAR_START} → {THIRD_PRIOR_SCHOOL_YEAR_END})")
    print(f"  Today: {TODAY}")

    # ---------------------------
    # AY rows config (3 rows); P2 compares to P3
    # ---------------------------
    AY_ROWS = [
        {
            "label": CURRENT_ACADEMIC_YEAR,
            "school_start": CURRENT_SCHOOL_YEAR_START,
            "school_end": ACADEMIC_YEAR_END,
            "ytd_to": TODAY,
            "compare_start": PRIOR_SCHOOL_YEAR_START,
            "compare_end": PRIOR_SCHOOL_YEAR_END,
            "compare_ytd_to": PRIOR_YEAR_TODAY
        },
        {
            "label": PRIOR_ACADEMIC_YEAR,
            "school_start": PRIOR_SCHOOL_YEAR_START,
            "school_end": PRIOR_SCHOOL_YEAR_END,
            "ytd_to": PRIOR_YEAR_TODAY,
            "compare_start": SECOND_PRIOR_SCHOOL_YEAR_START,
            "compare_end": SECOND_PRIOR_SCHOOL_YEAR_END,
            "compare_ytd_to": PRIOR2_YEAR_TODAY
        },
        {
            "label": SECOND_PRIOR_ACADEMIC_YEAR,
            "school_start": SECOND_PRIOR_SCHOOL_YEAR_START,
            "school_end": SECOND_PRIOR_SCHOOL_YEAR_END,
            "ytd_to": PRIOR2_YEAR_TODAY,
            "compare_start": THIRD_PRIOR_SCHOOL_YEAR_START,
            "compare_end": THIRD_PRIOR_SCHOOL_YEAR_END,
            "compare_ytd_to": PRIOR3_YEAR_TODAY
        }
    ]

    # ---------------------------
    # Per-AY builder (keeps your column names)
    # ---------------------------
    def build_fact_for_ay(
        df_sessions,
        df_student_master,
        *,
        school_start: str,
        ytd_to: str,
        label: str,
        compare_start: str | None,
        compare_end: str | None,
        compare_ytd_to: str | None,
        HOURS_PER_SESSION: int = 3
    ):
        # ---------- YTD ----------
        current_ytd = df_sessions.filter(
            (F.col("Date") >= F.lit(school_start)) & (F.col("Date") <= F.lit(ytd_to))
        ).groupBy("studentkey").agg(
            F.avg(F.col("is_present")).alias("current_ytd_attendance_pc"),
            F.sum(F.when(F.col("is_present") == 0, 1).otherwise(0)).alias("number_of_sessions_ytd_absent"),
            F.sum(F.col("is_possible")).alias("total_sessions_ytd"),
            F.sum(F.col("is_auth_abs")).alias("auth_absences_ytd"),
            F.sum(F.col("is_unauth_abs")).alias("unauth_absences_ytd")
        )

        if current_ytd.count() == 0:
            all_students = df_sessions.select("studentkey").distinct()
            current_ytd = all_students.select(
                "studentkey",
                F.lit(0.0).alias("current_ytd_attendance_pc"),
                F.lit(0).alias("number_of_sessions_ytd_absent"),
                F.lit(0).alias("total_sessions_ytd"),
                F.lit(0).alias("auth_absences_ytd"),
                F.lit(0).alias("unauth_absences_ytd")
            )

        # ---------- Compare windows for trend_PYTD ----------
        if compare_start and compare_ytd_to:
            prior_ytd = df_sessions.filter(
                (F.col("Date") >= F.lit(compare_start)) & (F.col("Date") <= F.lit(compare_ytd_to))
            ).groupBy("studentkey").agg(
                F.avg(F.col("is_present")).alias("prior_ytd_attendance_pc"),
                F.sum(F.when(F.col("is_present") == 0, 1).otherwise(0)).alias("number_of_sessions_PYTD_absent")
            )
        else:
            prior_ytd = current_ytd.select(
                "studentkey",
                F.lit(None).cast("double").alias("prior_ytd_attendance_pc"),
                F.lit(0).alias("number_of_sessions_PYTD_absent")
            )

        if compare_start and compare_end:
            prior_full = df_sessions.filter(
                (F.col("Date") >= F.lit(compare_start)) & (F.col("Date") <= F.lit(compare_end))
            ).groupBy("studentkey").agg(
                F.avg(F.col("is_present")).alias("prior_fullyear_attendance_pc")
            )
        else:
            prior_full = current_ytd.select(
                "studentkey",
                F.lit(None).cast("double").alias("prior_fullyear_attendance_pc")
            )

        # ---------- Rolling trends (unchanged) ----------
        student_last_dates = df_sessions.filter(F.col("is_possible") == 1).groupBy("studentkey").agg(
            F.max("Date").alias("last_date")
        )
        sessions_numbered = df_sessions.join(student_last_dates, on="studentkey").filter(F.col("is_possible") == 1)
        window_backwards = Window.partitionBy("studentkey").orderBy(F.desc("Date"), F.desc("Session"))
        sessions_numbered = sessions_numbered.withColumn("days_back_from_end", F.row_number().over(window_backwards) - 1)

        latest_trends = sessions_numbered.groupBy("studentkey").agg(
            F.avg(F.when(F.col("days_back_from_end") < 10, F.col("is_present"))).alias("att_last_7"),
            F.avg(F.when((F.col("days_back_from_end") >= 10) & (F.col("days_back_from_end") < 20), F.col("is_present"))).alias("att_prev_7"),
            F.avg(F.when(F.col("days_back_from_end") < 60, F.col("is_present"))).alias("att_last_30"),
            F.avg(F.when((F.col("days_back_from_end") >= 60) & (F.col("days_back_from_end") < 120), F.col("is_present"))).alias("att_prev_30"),
            F.sum(F.when(F.col("days_back_from_end") < 10, 1).otherwise(0)).alias("count_last_7"),
            F.sum(F.when((F.col("days_back_from_end") >= 10) & (F.col("days_back_from_end") < 20), 1).otherwise(0)).alias("count_prev_7"),
            F.sum(F.when(F.col("days_back_from_end") < 60, 1).otherwise(0)).alias("count_last_30"),
            F.sum(F.when((F.col("days_back_from_end") >= 60) & (F.col("days_back_from_end") < 120), 1).otherwise(0)).alias("count_prev_30")
        ).withColumn(
            "trend_7_pc",
            F.when((F.col("att_prev_7").isNotNull()) & (F.col("att_prev_7") > 0) & (F.col("count_prev_7") >= 8),
                (F.col("att_last_7") - F.col("att_prev_7")) / F.col("att_prev_7")).otherwise(0)
        ).withColumn(
            "trend_30_pc",
            F.when((F.col("att_prev_30").isNotNull()) & (F.col("att_prev_30") > 0) & (F.col("count_prev_30") >= 40),
                (F.col("att_last_30") - F.col("att_prev_30")) / F.col("att_prev_30")).otherwise(0)
        ).withColumn(
            "trend_velocity_7",
            F.when(F.col("att_prev_7").isNotNull(), (F.col("att_last_7") - F.col("att_prev_7")) / 10).otherwise(0)
        ).withColumn(
            "trend_velocity_30",
            F.when(F.col("att_prev_30").isNotNull(), (F.col("att_last_30") - F.col("att_prev_30")) / 60).otherwise(0)
        ).withColumn(
            "trend_7_description",
            F.when(F.col("count_prev_7") < 8, "Insufficient Data")
            .when(F.col("trend_7_pc") > 0.05, "Significantly Better")
            .when(F.col("trend_7_pc") > 0.03, "Moderately Better")
            .when(F.col("trend_7_pc") > 0.01, "Marginally Better")
            .when(F.col("trend_7_pc") >= -0.01, "Stable")
            .when(F.col("trend_7_pc") > -0.03, "Marginally Worse")
            .when(F.col("trend_7_pc") > -0.05, "Moderately Worse")
            .otherwise("Significantly Worse")
        ).withColumn(
            "trend_30_description",
            F.when(F.col("count_prev_30") < 40, "Insufficient Data")
            .when(F.col("trend_30_pc") > 0.05, "Significantly Better")
            .when(F.col("trend_30_pc") > 0.03, "Moderately Better")
            .when(F.col("trend_30_pc") > 0.01, "Marginally Better")
            .when(F.col("trend_30_pc") >= -0.01, "Stable")
            .when(F.col("trend_30_pc") > -0.03, "Marginally Worse")
            .when(F.col("trend_30_pc") > -0.05, "Moderately Worse")
            .otherwise("Significantly Worse")
        ).select(
            "studentkey",
            "att_last_7","att_prev_7","att_last_30","att_prev_30",
            "trend_7_pc","trend_30_pc","trend_velocity_7","trend_velocity_30",
            "trend_7_description","trend_30_description"
        )

        # ---------- Day-of-week within this row's YTD ----------
        dow_attendance = df_sessions.filter(
            (F.col("Date") >= F.lit(school_start)) & (F.col("Date") <= F.lit(ytd_to))
        ).withColumn("day_of_week", F.dayofweek("Date"))

        dow_pivot = dow_attendance.groupBy("studentkey","day_of_week").agg(
            F.avg(F.col("is_present")).alias("dow_attendance_rate"),
            F.sum(F.col("is_possible")).alias("dow_possible_sessions")
        ).groupBy("studentkey").pivot("day_of_week", [2,3,4,5,6]).agg(
            F.first("dow_attendance_rate").alias("attendance"),
            F.first("dow_possible_sessions").alias("sessions")
        )

        dow_stats = dow_pivot.select(
            "studentkey",
            F.coalesce(F.col("2_attendance"), F.lit(0)).alias("DoW_ytd_attendance_pc_Mon"),
            F.coalesce(F.col("3_attendance"), F.lit(0)).alias("DoW_ytd_attendance_pc_Tue"),
            F.coalesce(F.col("4_attendance"), F.lit(0)).alias("DoW_ytd_attendance_pc_Wed"),
            F.coalesce(F.col("5_attendance"), F.lit(0)).alias("DoW_ytd_attendance_pc_Thu"),
            F.coalesce(F.col("6_attendance"), F.lit(0)).alias("DoW_ytd_attendance_pc_Fri")
        ).withColumn(
            "dow_max", F.greatest("DoW_ytd_attendance_pc_Mon","DoW_ytd_attendance_pc_Tue",
                                "DoW_ytd_attendance_pc_Wed","DoW_ytd_attendance_pc_Thu",
                                "DoW_ytd_attendance_pc_Fri")
        ).withColumn(
            "dow_min", F.least("DoW_ytd_attendance_pc_Mon","DoW_ytd_attendance_pc_Tue",
                            "DoW_ytd_attendance_pc_Wed","DoW_ytd_attendance_pc_Thu",
                            "DoW_ytd_attendance_pc_Fri")
        ).withColumn(
            "Has_DoW_pattern", F.when((F.col("dow_max")-F.col("dow_min")) > 0.1, 1).otherwise(0)
        ).drop("dow_max","dow_min")

        # ---------- Risk metrics ----------
        risk_metrics = current_ytd.withColumn(
            "attendance_deficit",
            F.col("number_of_sessions_ytd_absent") - (F.col("total_sessions_ytd") * 0.1)
        ).withColumn(
            "sessions_to_PA",
            F.when(F.col("attendance_deficit") > 0, -F.ceil(F.col("attendance_deficit")))
            .otherwise(F.ceil((F.col("total_sessions_ytd") * 0.1) - F.col("number_of_sessions_ytd_absent")))
        ).withColumn(
            "sessions_to_SA",
            F.when(F.col("number_of_sessions_ytd_absent") > (F.col("total_sessions_ytd") * 0.5),
                -F.ceil(F.col("number_of_sessions_ytd_absent") - (F.col("total_sessions_ytd") * 0.5)))
            .otherwise(F.ceil((F.col("total_sessions_ytd") * 0.5) - F.col("number_of_sessions_ytd_absent")))
        ).select("studentkey","sessions_to_PA","sessions_to_SA")

        # ---------- Learning hours ----------
        learning_hours = df_sessions.groupBy("studentkey").agg(
            F.sum(F.when(
                (F.col("Date") >= F.lit(school_start)) & (F.col("Date") <= F.lit(ytd_to)) &
                (F.col("is_present") == 0) & (F.col("is_possible") == 1), 3
            ).otherwise(0)).alias("Hours_of_Lost_Learning_ytd"),

            F.sum(F.when(
                F.lit(compare_start).isNotNull() & F.lit(compare_end).isNotNull() &
                (F.col("Date") >= F.lit(compare_start)) & (F.col("Date") <= F.lit(compare_end)) &
                (F.col("is_present") == 0) & (F.col("is_possible") == 1), 3
            ).otherwise(0)).alias("Hours_of_Lost_Learning_PYTD"),

            F.sum(F.when((F.col("is_present") == 0) & (F.col("is_possible") == 1), 3).otherwise(0)).alias("Hours_of_Lost_Learning_Career"),
            F.min("Date").alias("first_attendance_date"),
            F.max("Date").alias("last_attendance_date"),
            F.countDistinct("Academic_Year").alias("years_of_data")
        )

        # ---------- Streaks within this AY (session-level) ----------
        window_streak = Window.partitionBy("studentkey").orderBy("Date","Session")
        streak_sessions = df_sessions.filter(F.col("Date") >= F.lit(school_start)).select(
            "studentkey","Date","Session","is_present","is_possible"
        ).filter(F.col("is_possible") == 1)

        streak_sessions = streak_sessions.withColumn("streak_break", F.when(F.col("is_present") == 0, 1).otherwise(0)) \
            .withColumn("streak_group", F.sum("streak_break").over(window_streak))

        window_streak_group = Window.partitionBy("studentkey","streak_group").orderBy("Date","Session")
        streak_sessions = streak_sessions.withColumn(
            "streak_count",
            F.when(F.col("is_present") == 1, F.row_number().over(window_streak_group)).otherwise(0)
        )

        window_last_session = Window.partitionBy("studentkey").orderBy(F.desc("Date"), F.desc("Session"))
        current_streaks = streak_sessions.withColumn("row_num", F.row_number().over(window_last_session)) \
            .filter(F.col("row_num") == 1).select("studentkey", F.col("streak_count").alias("Streak_session_count"))

        current_streaks = current_streaks.withColumn(
            "Streak_current_band",
            F.when(F.col("Streak_session_count") >= 100, F.concat(F.lit("Platinum ("), F.col("Streak_session_count"), F.lit(")"))) \
            .when(F.col("Streak_session_count") >= 40,  F.concat(F.lit("Gold ("), F.col("Streak_session_count"), F.lit(")"))) \
            .when(F.col("Streak_session_count") >= 20,  F.concat(F.lit("Silver ("), F.col("Streak_session_count"), F.lit(")"))) \
            .when(F.col("Streak_session_count") >= 10,  F.concat(F.lit("Bronze ("), F.col("Streak_session_count"), F.lit(")"))) \
            .otherwise(F.concat(F.lit("Getting Started ("), F.col("Streak_session_count"), F.lit(")")))
        )

        ytd_max_streaks = streak_sessions.filter(F.col("Date") >= F.lit(school_start)).groupBy("studentkey").agg(
            F.max("streak_count").alias("max_streak_ytd")
        ).withColumn(
            "Streak_highest_band_ytd",
            F.when(F.col("max_streak_ytd") >= 100, F.concat(F.lit("Platinum ("), F.col("max_streak_ytd"), F.lit(")"))) \
            .when(F.col("max_streak_ytd") >= 40,  F.concat(F.lit("Gold ("), F.col("max_streak_ytd"), F.lit(")"))) \
            .when(F.col("max_streak_ytd") >= 20,  F.concat(F.lit("Silver ("), F.col("max_streak_ytd"), F.lit(")"))) \
            .when(F.col("max_streak_ytd") >= 10,  F.concat(F.lit("Bronze ("), F.col("max_streak_ytd"), F.lit(")"))) \
            .otherwise(F.concat(F.lit("Getting Started ("), F.col("max_streak_ytd"), F.lit(")")))
        )

        # ---------- Absence metrics ----------
        current_year_check = df_sessions.filter(F.col("Date") >= F.lit(school_start)).count()
        if current_year_check == 0:
            all_students = df_sessions.select("studentkey").distinct()
            actual_absences_sessions = all_students.select(
                "studentkey",
                F.lit(0).alias("sessions_actual_abs_ytd"),
                F.lit(0).alias("sessions_actual_abs_career_placeholder")
            )
            actual_absences_days = all_students.select(
                "studentkey",
                F.lit(0).alias("days_actual_abs_ytd"),
                F.lit(0).alias("days_actual_abs_career_placeholder")
            )

            career_absences_sessions = df_sessions.filter(F.col("is_possible") == 1).groupBy("studentkey").agg(
                F.sum(F.when(F.col("is_present") == 0, 1).otherwise(0)).alias("sessions_actual_abs_career"),
                F.sum(F.when(
                    F.lit(compare_start).isNotNull() & F.lit(compare_end).isNotNull() &
                    (F.col("Date") >= F.lit(compare_start)) & (F.col("Date") <= F.lit(compare_end)) &
                    (F.col("is_present") == 0), 1
                ).otherwise(0)).alias("sessions_actual_abs_prior_year")
            )

            career_absences_days = df_sessions.filter((F.col("is_possible") == 1) & (F.col("Session") == "AM")).groupBy("studentkey").agg(
                F.sum(F.when(F.col("is_present") == 0, 1).otherwise(0)).alias("days_actual_abs_career"),
                F.sum(F.when(
                    F.lit(compare_start).isNotNull() & F.lit(compare_end).isNotNull() &
                    (F.col("Date") >= F.lit(compare_start)) & (F.col("Date") <= F.lit(compare_end)) &
                    (F.col("is_present") == 0), 1
                ).otherwise(0)).alias("days_actual_abs_prior_year")
            )

            actual_absences_sessions = actual_absences_sessions.drop("sessions_actual_abs_career_placeholder") \
                .join(career_absences_sessions, "studentkey", "left")
            actual_absences_days = actual_absences_days.drop("days_actual_abs_career_placeholder") \
                .join(career_absences_days, "studentkey", "left")
            actual_absences = actual_absences_sessions.join(actual_absences_days, "studentkey", "left")

        else:
            actual_absences_sessions = df_sessions.filter(F.col("is_possible") == 1).groupBy("studentkey").agg(
                F.sum(F.when(
                    (F.col("Date") >= F.lit(school_start)) & (F.col("Date") < F.lit(ytd_to)) & (F.col("is_present") == 0), 1
                ).otherwise(0)).alias("sessions_actual_abs_ytd"),
                F.sum(F.when(F.col("is_present") == 0, 1).otherwise(0)).alias("sessions_actual_abs_career"),
                F.sum(F.when(
                    F.lit(compare_start).isNotNull() & F.lit(compare_end).isNotNull() &
                    (F.col("Date") >= F.lit(compare_start)) & (F.col("Date") <= F.lit(compare_end)) &
                    (F.col("is_present") == 0), 1
                ).otherwise(0)).alias("sessions_actual_abs_prior_year")
            )

            actual_absences_days = df_sessions.filter((F.col("is_possible") == 1) & (F.col("Session") == "AM")).groupBy("studentkey").agg(
                F.sum(F.when(
                    (F.col("Date") >= F.lit(school_start)) & (F.col("Date") < F.lit(ytd_to)) & (F.col("is_present") == 0), 1
                ).otherwise(0)).alias("days_actual_abs_ytd"),
                F.sum(F.when(F.col("is_present") == 0, 1).otherwise(0)).alias("days_actual_abs_career"),
                F.sum(F.when(
                    F.lit(compare_start).isNotNull() & F.lit(compare_end).isNotNull() &
                    (F.col("Date") >= F.lit(compare_start)) & (F.col("Date") <= F.lit(compare_end)) &
                    (F.col("is_present") == 0), 1
                ).otherwise(0)).alias("days_actual_abs_prior_year")
            )

            actual_absences = actual_absences_sessions.join(actual_absences_days, "studentkey", "left")

        # ---------- Last present/absent (relative to ytd_to) ----------
        window_last = Window.partitionBy("studentkey").orderBy(F.desc("Date"), F.desc("Session"))
        last_present = df_sessions.filter((F.col("is_present") == 1) & (F.col("is_possible") == 1)).withColumn(
            "row_num", F.row_number().over(window_last)
        ).filter(F.col("row_num") == 1).select(
            "studentkey", F.datediff(F.lit(ytd_to), F.col("Date")).alias("days_since_last_present")
        )
        last_absent = df_sessions.filter((F.col("is_present") == 0) & (F.col("is_possible") == 1)).withColumn(
            "row_num", F.row_number().over(window_last)
        ).filter(F.col("row_num") == 1).select(
            "studentkey", F.datediff(F.lit(ytd_to), F.col("Date")).alias("days_since_last_abs")
        )

        # ---------- Latest Is_10_in_10 within this AY ----------
        window_latest_ay = Window.partitionBy("studentkey").orderBy(F.desc("Date"), F.desc("Session"))

        latest_is_10_in_10 = (
            df_sessions
            .filter(
                (F.col("Academic_Year") == F.lit(label)) &
                (F.col("is_possible") == 1)  # keep if you only want valid sessions; remove if not desired
            )
            .withColumn("rn", F.row_number().over(window_latest_ay))
            .filter(F.col("rn") == 1)
            .select(
                "studentkey",
                F.col("Is_10_in_10").alias("Is_10_in_10")   # keep the original column name in the output
            )
        )

        # ---------- Assemble ----------
        fact_row = current_ytd \
            .join(prior_ytd, "studentkey", "left") \
            .join(prior_full, "studentkey", "left") \
            .join(risk_metrics, "studentkey", "left") \
            .join(dow_stats, "studentkey", "left") \
            .join(learning_hours, "studentkey", "left") \
            .join(current_streaks, "studentkey", "left") \
            .join(ytd_max_streaks.select("studentkey","Streak_highest_band_ytd"), "studentkey", "left") \
            .join(actual_absences, "studentkey", "left") \
            .join(last_present, "studentkey", "left") \
            .join(last_absent, "studentkey", "left") \
            .join(latest_trends, "studentkey", "left")\
            .join(latest_is_10_in_10, "studentkey", "left")

        fact_row = fact_row.withColumn(
            "trend_PYTD_pc",
            F.when(F.col("prior_ytd_attendance_pc") > 0,
                (F.col("current_ytd_attendance_pc") - F.col("prior_ytd_attendance_pc")) / F.col("prior_ytd_attendance_pc"))
            .otherwise(0)
        ).withColumn(
            "trend_PYTD_description",
            F.when(F.col("trend_PYTD_pc") > 5, "Significantly Better")
            .when(F.col("trend_PYTD_pc") > 3, "Moderately Better")
            .when(F.col("trend_PYTD_pc") > 1, "Marginally Better")
            .when(F.col("trend_PYTD_pc") > -1, "Similar")
            .when(F.col("trend_PYTD_pc") > -3, "Marginally Worse")
            .when(F.col("trend_PYTD_pc") > -5, "Moderately Worse")
            .otherwise("Significantly Worse")
        ).withColumn(
            "trend_velocity_PYTD",
            F.when(F.lit(school_start).isNotNull() & F.lit(ytd_to).isNotNull(),
                (F.col("current_ytd_attendance_pc") - F.col("prior_ytd_attendance_pc")) /
                F.greatest(F.lit(1), F.datediff(F.lit(ytd_to), F.lit(school_start)))
            ).otherwise(F.lit(0))
        ).withColumn("last_updated", F.lit(datetime.now())) \
        .withColumn("Academic_Year", F.lit(label))

        return fact_row

    # ---------------------------
    # Build three AY rows & save
    # ---------------------------
    AY_ROWS = AY_ROWS  # already defined above

    rows = []
    for ay in AY_ROWS:
        row_df = build_fact_for_ay(
            df_sessions=df_sessions,
            df_student_master=df_student_master,
            school_start=ay["school_start"],
            ytd_to=ay["ytd_to"],
            label=ay["label"],
            compare_start=ay["compare_start"],
            compare_end=ay["compare_end"],
            compare_ytd_to=ay["compare_ytd_to"],
        )
        rows.append(row_df)

    fact_trends_all = rows[0].unionByName(rows[1], allowMissingColumns=True).unionByName(rows[2], allowMissingColumns=True)

    fact_trends_all = add_student_academic_year_group_key(fact_trends_all)
    try:
        fact_trends_all = add_student_ext_academic_year_key(fact_trends_all)
    except Exception as e:
            print(f"Error adding student_ext_academic_year_key {e}")
    
    from pyspark.sql import functions as F

    # list of columns to null out when Academic_Year != CURRENT_ACADEMIC_YEAR as they are not relevant
    cols_to_null = [
        "days_since_last_present",
        "days_since_last_abs",
        "att_last_7",
        "att_prev_7",
        "att_last_30",
        "att_prev_30",
        "trend_7_pc",
        "trend_30_pc",
        "trend_velocity_7",
        "trend_velocity_30",
        "trend_7_description",
        "trend_30_description",
        "Is_10_in_10"
    ]

    # apply transformation
    fact_trends_all = fact_trends_all.select(
        *[
            F.when(
                F.col("Academic_Year") == CURRENT_ACADEMIC_YEAR, F.col(c)
            ).otherwise(F.lit(None)).alias(c)
            if c in cols_to_null else F.col(c)
            for c in fact_trends_all.columns
        ]
    )

    # Save
    output_path = os.path.join(gold_path, "fact_AttendanceTrends")
    fact_trends_all.write.mode("overwrite").parquet(output_path)

    # --- Row counts per Academic_Year ---
    counts_df = (
        fact_trends_all
        .groupBy("Academic_Year")
        .count()
        .orderBy("Academic_Year")
    )

    print("\nRow counts by Academic_Year:")
    for r in counts_df.collect():
        print(f"  {r['Academic_Year']}: {r['count']:,}")

    print(f"\nTotal rows written: {fact_trends_all.count():,}")

    print(f"Saved {fact_trends_all.count():,} rows (one row per student per AY: Current, Prior, Second-Prior) to {output_path}")
    print("="*60)
    print("fact_AttendanceTrends BUILD COMPLETE")
    print("="*60)


except Exception as e:
    print("Skipping fact_AttendanceTrends due to an error")

oeai.log.end_block()
oeai.log.checkpoint()

In [0]:
oeai.log.end_block(index=0, include_target=True)
oeai.log.shutdown()
oeai.log.checkpoint()